# CinnaAI — Two-Class Bark Disease Training in Google Colab (v5)

**Use the dataset you already have:** `RoughBark` and `StripeCanker`. No extra dataset is required for the default workflow. No healthy class is invented.

## Default decision flow: reference_only_experimental

1. Decode EXIF-corrected RGB, resize to 224×224, float32 `[0,255]`; no external `/255`.
2. Compare its embedding with unaugmented disease-training references. Outside the validation-derived threshold → `uncertain`, with no disease classification.
3. Inside the reference → run the two-disease classifier and display class scores.
4. Reference pass + validation-derived disease confidence/margin pass → `experimental_candidate`, with the predicted disease name. Otherwise → `uncertain`, with diagnostic class scores.

Missing relevance data no longer blocks disease classification in the default mode. `experimental_candidate` is an explicitly experimental result, not an accepted diagnosis. It always recommends review and keeps `input_relevance_verified=false`, `healthy_detection_supported=false` and `is_cinnamon=null`.

Reference distance can screen some unrelated images, but its ability to reject unseen objects, healthy bark or unknown diseases is unmeasured without representative negative tests. Some real bark can also fall outside the reference. Do not raise thresholds simply to force an inspected image through.

## Running and updating

Choose a Colab GPU and run top-to-bottom. Only missing packages are installed; the current Colab TensorFlow/Keras stack is preserved. All 97 original numbered sections and the existing labelled subsections remain in order. EfficientNetB0, frozen-backbone Stage 1, selective fine-tuning Stage 2, validation selection and held-out disease evaluation are preserved.

**Already trained in an active session?** Replace and rerun CELL 44B and CELL 44I from this notebook, then rerun CELLS 60, 96 and 97. CELL 44I sets the new mode on the existing policy. No disease retraining is needed for that inference fix. Enable `RUN_OPTIONAL_MANUAL_TEST=True` in CELL 96. The two reported images have different reference outcomes: `0.85817 > 0.27865` stops; `0.07838 < 0.27865` reaches classification. The bark image's class/confidence is determined by the trained classifier, not by these distances alone.

To regenerate a consistent export, rerun CELLS 52–94 in order after the fix, then enable Drive backup in CELL 94A. CELL 64 recreates the export directory, so preserve any earlier package you need first. Recomputed metrics, runtime, inference mode, both model formats, policy/reference, parity fixtures, checksums and ZIP travel together. CELLS 96–97 and Drive mounting remain opt-in for unattended Run all.

## Optional relevance extension

CELLS 44F–44H skip cleanly when `/content/bark_relevance_dataset` is absent. They are optional, not a data-collection prerequisite. If a relevance model is later supplied, its calibrated rejection or uncertainty still takes priority; the fallback never bypasses an inconclusive trained model. `INFERENCE_MODE="staged_relevance"` in CELL 44A restores strict relevance requirements. Old policy files with no mode remain strict for compatibility.

The optional negative folders in CELL 44A are evaluation/calibration inputs only. No unrelated examples or healthy disease labels are fabricated. Historical v4 notes describe the old strict-only behavior; this v5 overview and the executed sources supersede them. Neither successful export parity nor tests of the code establish field accuracy or tree/site independence.


## CELL 1 — Check GPU

**Repair record:** Preserved the supplied Colab shell command.


In [ ]:
!nvidia-smi


## CELL 2 — Install only missing packages

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
import sys
import subprocess
import importlib.util

REQUIRED_PACKAGES = {
    "kagglehub": "kagglehub", "tensorflow": "tensorflow", "keras": "keras",
    "numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib",
    "sklearn": "scikit-learn", "PIL": "Pillow", "h5py": "h5py",
}
missing_packages = [pip_name for module_name, pip_name in REQUIRED_PACKAGES.items()
                    if importlib.util.find_spec(module_name) is None]
if missing_packages:
    print("Installing missing packages:", missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
else:
    print("All required packages are already available. Keeping the current Colab stack.")


## CELL 3 — Imports

**Repair record:** Restored imports; added importlib.metadata to record installed package versions without changing packages.


In [ ]:
import os
import sys
import json
import math
import shutil
import hashlib
import random
import platform
from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata as package_metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display
import tensorflow as tf
import keras
import sklearn
import PIL
import h5py
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.utils.class_weight import compute_class_weight


## CELL 4 — Print and record Colab environment

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
print("=" * 70)
print("TRAINING ENVIRONMENT")
print("=" * 70)
for name, version in {
    "Python": platform.python_version(), "TensorFlow": tf.__version__,
    "Keras": keras.__version__, "NumPy": np.__version__, "Pandas": pd.__version__,
    "Scikit-learn": sklearn.__version__, "Pillow": PIL.__version__, "h5py": h5py.__version__,
}.items():
    print(f"{name:14s}: {version}")
print("GPU devices:", tf.config.list_physical_devices("GPU"))
if not tf.config.list_physical_devices("GPU"):
    print("WARNING: TensorFlow cannot see a GPU. Select a GPU runtime before training.")
else:
    print("PASS: TensorFlow GPU detected.")


## CELL 5 — Reproducibility settings

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
    print("Deterministic TensorFlow operations enabled.")
except Exception as e:
    print("Could not fully enable deterministic mode:", e)


## CELL 6 — Download dataset from Kaggle

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
KAGGLE_DATASET_ID = (
    "madhavipethangoda/"
    "cinnamon-plant-stem-and-branch-disease-dataset"
)
dataset_path = kagglehub.dataset_download(KAGGLE_DATASET_ID)
DATASET_ROOT = Path(dataset_path)
print("Dataset downloaded to:", DATASET_ROOT)


## CELL 7 — Display dataset structure

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
print("Dataset root:", DATASET_ROOT)
print("Top-level contents:")
for item in sorted(DATASET_ROOT.iterdir()):
    item_type = "DIR" if item.is_dir() else "FILE"
    print(f"[{item_type}] {item.name}")


## CELL 8 — Automatically locate the Diseases folder

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
disease_folder_candidates = []
for candidate in DATASET_ROOT.rglob("Diseases"):
    if candidate.is_dir() and (candidate / "RoughBark").is_dir() and (candidate / "StripeCanker").is_dir():
        disease_folder_candidates.append(candidate)
disease_folder_candidates = sorted(disease_folder_candidates)
print("Matching Diseases folders:")
for candidate in disease_folder_candidates:
    print(candidate)
if not disease_folder_candidates:
    raise FileNotFoundError("Could not locate a Diseases folder containing both RoughBark and StripeCanker.")
DATASET_DIR = disease_folder_candidates[0]
print("Selected dataset directory:", DATASET_DIR)


## CELL 9 — Define the exact model class order

`output[0] = RoughBark`; `output[1] = StripeCanker`. Never change this order after training.

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
CLASS_NAMES = ["RoughBark", "StripeCanker"]
CLASS_TO_ID = {class_name: class_id for class_id, class_name in enumerate(CLASS_NAMES)}
CLASS_DIRS = {class_name: DATASET_DIR / class_name for class_name in CLASS_NAMES}
print("Class mapping:", CLASS_TO_ID)
for class_name, class_path in CLASS_DIRS.items():
    print(class_name, "->", class_path, "| Exists:", class_path.is_dir())
    if not class_path.is_dir():
        raise FileNotFoundError(class_path)


## CELL 10 — Build image manifest

**Repair record:** Completed the class-count display and supplied explicit columns so an empty manifest receives a clear validation error.


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
records = []
for class_id, class_name in enumerate(CLASS_NAMES):
    for image_path in sorted(CLASS_DIRS[class_name].rglob("*")):
        if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
            records.append({"path": str(image_path), "filename": image_path.name,
                            "class_id": class_id, "class_name": class_name})
df = pd.DataFrame(records, columns=["path", "filename", "class_id", "class_name"])
print("Total images found:", len(df))
print("Images per class:")
display(df["class_name"].value_counts().reindex(CLASS_NAMES, fill_value=0))


## CELL 11 — Make sure both classes contain enough images

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
class_counts = df["class_name"].value_counts().reindex(CLASS_NAMES, fill_value=0)
print(class_counts)
for class_name in CLASS_NAMES:
    count = int(class_counts[class_name])
    if count < 20:
        raise RuntimeError(f"{class_name} has only {count} images. This is too small for the planned split.")
print("PASS: both classes contain enough images for the planned split; this is not a field-validity guarantee.")


## CELL 12 — Validate every image

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
def validate_image(image_path):
    try:
        with Image.open(image_path) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            image.load()
        return True, None
    except Exception as e:
        return False, str(e)

validation_records = []
for _, row in df.iterrows():
    valid, error = validate_image(row["path"])
    validation_records.append({"path": row["path"], "valid": valid, "error": error})
image_validation_df = pd.DataFrame(validation_records)
bad_images = image_validation_df[~image_validation_df["valid"]].copy()
print("Invalid images:", len(bad_images))
if not bad_images.empty:
    display(bad_images)


## CELL 13 — Remove invalid images

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
valid_paths = set(image_validation_df.loc[image_validation_df["valid"], "path"])
df = df[df["path"].isin(valid_paths)].reset_index(drop=True)
print("Usable images:", len(df))
print(df["class_name"].value_counts().reindex(CLASS_NAMES, fill_value=0))


## CELL 14 — Calculate SHA-256 hashes

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
def sha256_file(file_path):
    hasher = hashlib.sha256()
    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

df["sha256"] = [sha256_file(file_path) for file_path in df["path"]]
print("SHA-256 hashes calculated.")


## CELL 15 — Find exact duplicates

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
duplicate_rows = df.groupby("sha256").filter(lambda group: len(group) > 1).sort_values("sha256")
print("Rows involved in exact duplicates:", len(duplicate_rows))
if len(duplicate_rows):
    display(duplicate_rows[["filename", "class_name", "path", "sha256"]])


## CELL 16 — Detect duplicate images with conflicting labels

All copies of an identical file with contradictory disease labels are excluded from the manifest. The original dataset files are retained. No folder-order label is assumed.

**Repair record:** Exclude all copies of cross-label identical files from the in-memory manifest, preserve originals, and retain an audit table instead of stopping or selecting a label arbitrarily.


In [ ]:
duplicate_class_counts = df.groupby("sha256")["class_name"].nunique()
conflicting_hashes = duplicate_class_counts[duplicate_class_counts > 1].index.tolist()
conflicting_images = df[df["sha256"].isin(conflicting_hashes)].sort_values("sha256").copy()
CONFLICTING_HASH_GROUPS_REMOVED = len(conflicting_hashes)
CONFLICTING_IMAGES_REMOVED = len(conflicting_images)
print("Identical files with conflicting labels:", CONFLICTING_HASH_GROUPS_REMOVED)
if conflicting_hashes:
    display(conflicting_images)
    # Exclude every copy: folder order cannot establish the correct disease label.
    df = df[~df["sha256"].isin(conflicting_hashes)].reset_index(drop=True)
    print("Excluded all conflicting copies from training/evaluation:", CONFLICTING_IMAGES_REMOVED)
    print("Original files are untouched; excluded rows will be saved in the handoff audit.")
else:
    print("PASS: no conflicting exact duplicates.")
assert not (df.groupby("sha256")["class_name"].nunique() > 1).any()
print("Remaining class counts:")
print(df["class_name"].value_counts().reindex(CLASS_NAMES, fill_value=0))


## CELL 17 — Remove duplicate copies

Only same-label exact duplicates may be reduced to one copy. CELL 16 must exclude cross-label conflicts first.

**Repair record:** Completed duplicate removal and post-cleaning class checks; added a pre-deduplication guard so CELL 17 cannot silently choose a label for an unresolved cross-label file.


In [ ]:
assert not (df.groupby("sha256")["class_name"].nunique() > 1).any(), "Run CELL 16 before deduplicating."
before_deduplication = len(df)
df = (df.sort_values(["class_id", "path"]).drop_duplicates(subset=["sha256"], keep="first")
      .reset_index(drop=True))
after_deduplication = len(df)
DUPLICATES_REMOVED = before_deduplication - after_deduplication
print("Images before:", before_deduplication)
print("Images after:", after_deduplication)
print("Exact duplicates removed:", DUPLICATES_REMOVED)
for class_name in CLASS_NAMES:
    count = int((df["class_name"] == class_name).sum())
    if count < 20:
        raise RuntimeError(f"After cleaning, {class_name} has only {count} images; collect/review data before splitting.")


## CELL 18 — Near-duplicate audit

This audit only reports suspicious pairs. Review them; no images are automatically deleted or grouped.

**Repair record:** The source only defined a hash function. Added candidate comparison, a results table, and explicit resource-cap/completeness reporting; no automatic deletion.


In [ ]:
def difference_hash(image_path, hash_size=8):
    with Image.open(image_path) as image:
        image = ImageOps.exif_transpose(image).convert("L")
        image = image.resize((hash_size + 1, hash_size), Image.Resampling.BILINEAR)
        pixels = np.asarray(image)
    differences = pixels[:, 1:] > pixels[:, :-1]
    hash_value = 0
    for index, value in enumerate(differences.flatten()):
        if value:
            hash_value |= 1 << index
    return hash_value

df["dhash"] = [f"{difference_hash(path):016x}" for path in df["path"]]
# Five disjoint bit chunks find candidate pairs at Hamming distance <= 4.
# No files are removed or relabelled by this heuristic audit.
NEAR_DUPLICATE_MAX_DISTANCE = 4
MAX_AUDIT_COMPARISONS = 2_000_000
MAX_AUDIT_RESULTS = 10_000
buckets = {}
near_duplicate_records = []
audit_comparisons = 0
near_duplicate_audit_complete = True
hash_values = [int(value, 16) for value in df["dhash"]]
chunks = [(0, 13), (13, 13), (26, 13), (39, 13), (52, 12)]
for right_index, right_hash in enumerate(hash_values):
    keys = [(part, (right_hash >> offset) & ((1 << width) - 1))
            for part, (offset, width) in enumerate(chunks)]
    candidates = set()
    for key in keys:
        candidates.update(buckets.get(key, []))
    for left_index in sorted(candidates):
        if audit_comparisons >= MAX_AUDIT_COMPARISONS or len(near_duplicate_records) >= MAX_AUDIT_RESULTS:
            near_duplicate_audit_complete = False
            break
        audit_comparisons += 1
        distance = (hash_values[left_index] ^ right_hash).bit_count()
        if distance <= NEAR_DUPLICATE_MAX_DISTANCE:
            near_duplicate_records.append({
                "path_a": df.iloc[left_index]["path"], "path_b": df.iloc[right_index]["path"],
                "class_a": df.iloc[left_index]["class_name"], "class_b": df.iloc[right_index]["class_name"],
                "hamming_distance": distance,
            })
    if not near_duplicate_audit_complete:
        break
    for key in keys:
        buckets.setdefault(key, []).append(right_index)
near_duplicate_df = pd.DataFrame(near_duplicate_records,
    columns=["path_a", "path_b", "class_a", "class_b", "hamming_distance"])
print("Potential near-duplicate pairs:", len(near_duplicate_df))
print("Heuristic audit complete:", near_duplicate_audit_complete)
if not near_duplicate_df.empty:
    display(near_duplicate_df.head(30))
    print("Review these pairs. They are not automatically removed or grouped; split leakage may remain.")
if not near_duplicate_audit_complete:
    print("Audit hit its resource cap; recorded pairs are a lower bound, not a complete count.")


## CELL 19 — Plot class distribution

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
class_counts = df["class_name"].value_counts().reindex(CLASS_NAMES)
display(class_counts.rename("image_count"))
plt.figure(figsize=(7, 5))
class_counts.plot(kind="bar")
plt.title("Cinnamon Bark Disease Class Distribution")
plt.xlabel("Disease class")
plt.ylabel("Number of images")
plt.xticks(rotation=0)
plt.show()


## CELL 20 — Display random images from each class

Inspect the image labels manually. Stop if visible label errors need correction.

**Repair record:** Completed the truncated image display loop and hid unused axes.


In [ ]:
fig, axes = plt.subplots(len(CLASS_NAMES), 5, figsize=(16, 7), squeeze=False)
for class_index, class_name in enumerate(CLASS_NAMES):
    class_df = df[df["class_name"] == class_name]
    samples = class_df.sample(n=min(5, len(class_df)), random_state=SEED)
    for column_index, (_, row) in enumerate(samples.iterrows()):
        with Image.open(row["path"]) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            axes[class_index, column_index].imshow(image)
        axes[class_index, column_index].set_title(class_name)
    for axis in axes[class_index]:
        axis.axis("off")
plt.tight_layout()
plt.show()


## CELL 21 — Create working directory

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
WORK_DIR = Path("/content/bark_training")
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Working directory:", WORK_DIR)


## CELL 22 — Create 70/15/15 train-validation-test split

Tree/site IDs are unavailable to this workflow. Related images can remain across partitions despite exact-file deduplication.

**Repair record:** Completed test partition reporting; retained the supplied image-level split and its independence limitation.


In [ ]:
train_df, temporary_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df["class_id"])
validation_df, test_df = train_test_split(temporary_df, test_size=0.50, random_state=SEED,
                                        stratify=temporary_df["class_id"])
train_df, validation_df, test_df = train_df.copy(), validation_df.copy(), test_df.copy()
train_df["split"], validation_df["split"], test_df["split"] = "train", "validation", "test"
for name, partition in [("TRAIN", train_df), ("VALIDATION", validation_df), ("TEST", test_df)]:
    print(name, "images:", len(partition))
    print(partition["class_name"].value_counts().reindex(CLASS_NAMES, fill_value=0))
    assert set(partition["class_id"]) == {0, 1}, f"Missing class in {name}"
print("Image-level stratified split. Tree/site independence has not been established.")


## CELL 23 — Verify no exact duplicate leakage

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
train_hashes = set(train_df["sha256"])
validation_hashes = set(validation_df["sha256"])
test_hashes = set(test_df["sha256"])
assert train_hashes.isdisjoint(validation_hashes)
assert train_hashes.isdisjoint(test_hashes)
assert validation_hashes.isdisjoint(test_hashes)
print("PASS: no exact duplicate file hashes cross partitions.")
print("This check does not exclude differently encoded or related images.")


## CELL 24 — Save permanent split manifest

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
split_manifest = pd.concat([train_df, validation_df, test_df], ignore_index=True)
SPLIT_MANIFEST_PATH = WORK_DIR / "split_manifest.csv"
split_manifest.to_csv(SPLIT_MANIFEST_PATH, index=False)
print("Saved:", SPLIT_MANIFEST_PATH)


## CELL 25 — Define exact bark preprocessing

EXIF correction → RGB → 224×224 Pillow bilinear → float32 `[0,255]`. **No external `/255`.**

**Repair record:** Completed the missing tf_load_image body with static shape and label dtype; preserved EXIF/RGB/Pillow bilinear/[0,255].


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 16

def load_image_pillow(path_tensor):
    path = path_tensor.numpy().decode("utf-8")
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image).convert("RGB")
        image = image.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
        array = np.asarray(image, dtype=np.float32)
    return array

def tf_load_image(path, label):
    image = tf.py_function(load_image_pillow, [path], Tout=tf.float32)
    image.set_shape((IMG_SIZE, IMG_SIZE, 3))
    return image, tf.cast(label, tf.int32)


## CELL 26 — Define training-only augmentation

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip(mode="horizontal", seed=SEED),
    tf.keras.layers.RandomRotation(factor=0.04, fill_mode="reflect", seed=SEED),
    tf.keras.layers.RandomZoom(height_factor=(-0.08, 0.08), width_factor=(-0.08, 0.08),
                               fill_mode="reflect", seed=SEED),
    tf.keras.layers.RandomTranslation(height_factor=0.05, width_factor=0.05,
                                      fill_mode="reflect", seed=SEED),
    tf.keras.layers.RandomContrast(factor=0.10, seed=SEED),
], name="training_augmentation")


## CELL 27 — Build TensorFlow datasets

**Repair record:** Completed batching, training-only shuffle/augmentation and all three dataset definitions. Used disk cache and clipped augmented pixels to [0,255].

**Staged relevance update:** Include label IDs in disk-cache keys so disease and relevance tasks cannot accidentally reuse cached targets.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(dataframe, training=False):
    paths = dataframe["path"].astype(str).to_numpy()
    labels = dataframe["class_id"].astype(np.int32).to_numpy()
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    dataset = dataset.map(tf_load_image, num_parallel_calls=AUTOTUNE)
    # Disk caching keeps decoded float32 images out of Colab RAM.
    cache_key = hashlib.sha256("\n".join(f"{path}\t{int(label)}" for path,label in zip(paths,labels)).encode("utf-8")).hexdigest()[:16]
    dataset = dataset.cache(str(WORK_DIR / f"decoded_{cache_key}"))
    if training:
        dataset = dataset.shuffle(len(dataframe), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.batch(BATCH_SIZE)
    if training:
        dataset = dataset.map(
            lambda images, labels: (tf.clip_by_value(augmentation(images, training=True), 0.0, 255.0), labels),
            num_parallel_calls=AUTOTUNE,
        )
    return dataset.prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
validation_ds = make_dataset(validation_df)
test_ds = make_dataset(test_df)
print("Training, validation and test pipelines created; test pixels have not been evaluated.")


## CELL 28 — Verify model input tensor

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
images, labels = next(iter(train_ds))
print("Batch shape:", images.shape, "dtype:", images.dtype)
print("Minimum pixel:", float(tf.reduce_min(images)))
print("Maximum pixel:", float(tf.reduce_max(images)))
assert tuple(images.shape[1:]) == (224, 224, 3)
assert images.dtype == tf.float32
assert float(tf.reduce_min(images)) >= 0.0 and float(tf.reduce_max(images)) <= 255.0


## CELL 29 — Display augmented training samples

Inspect these images to ensure augmentation preserves the labelled visual evidence.

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
sample_images, sample_labels = next(iter(train_ds))
plt.figure(figsize=(15, 8))
for index in range(min(10, len(sample_images))):
    plt.subplot(2, 5, index + 1)
    plt.imshow(np.clip(sample_images[index].numpy(), 0, 255).astype(np.uint8))
    plt.title(CLASS_NAMES[int(sample_labels[index])])
    plt.axis("off")
plt.tight_layout()
plt.show()


## CELL 30 — Calculate class weights

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
training_class_ids = np.array(sorted(train_df["class_id"].unique()))
calculated_weights = compute_class_weight(class_weight="balanced", classes=training_class_ids,
                                          y=train_df["class_id"].values)
class_weight = {int(class_id): float(weight) for class_id, weight in zip(training_class_ids, calculated_weights)}
print("Class weights:")
for class_id, weight in class_weight.items():
    print(CLASS_NAMES[class_id], "->", weight)


## CELL 31 — Build EfficientNetB0 model

**Repair record:** Completed the missing pooling/dropout/two-output softmax head and model construction. No augmentation or external rescaling in the inference graph.


In [ ]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32, name="bark_image")
backbone = tf.keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3),
)
BACKBONE_NAME = backbone.name
backbone.trainable = False
x = backbone(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
x = tf.keras.layers.Dropout(0.30, name="classification_dropout")(x)
outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax", dtype="float32",
                                name="bark_predictions")(x)
model = tf.keras.Model(inputs, outputs, name="cinnamon_bark_disease_model")
model.summary()


## CELL 32 — Verify model input/output contract

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)
print("Output dtype:", model.output.dtype)
assert model.input_shape == (None, 224, 224, 3)
assert model.output_shape == (None, 2)
assert model.output.dtype == "float32"
assert CLASS_NAMES == ["RoughBark", "StripeCanker"]
print("PASS: CinnaAI model contract satisfied.")


## CELL 33 — Compile Stage 1

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)


## CELL 34 — Stage 1 callbacks

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
STAGE1_PATH = WORK_DIR / "stage1_best.keras"
stage1_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(str(STAGE1_PATH), monitor="val_loss", mode="min",
                                       save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=6,
                                     restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", mode="min", factor=0.3,
                                        patience=2, min_lr=1e-6, verbose=1),
]


## CELL 35 — Train Stage 1

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
history_stage1 = model.fit(train_ds, validation_data=validation_ds, epochs=25,
                           class_weight=class_weight, callbacks=stage1_callbacks, verbose=1)


## CELL 36 — Prediction helper

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
def get_predictions(trained_model, dataframe, dataset):
    probabilities = np.asarray(trained_model.predict(dataset, verbose=0), dtype=np.float32)
    assert probabilities.shape == (len(dataframe), len(CLASS_NAMES))
    assert np.isfinite(probabilities).all()
    assert (probabilities >= 0.0).all() and (probabilities <= 1.0).all()
    np.testing.assert_allclose(probabilities.sum(axis=1), 1.0, atol=1e-5, rtol=0)
    predicted_ids = np.argmax(probabilities, axis=1)
    true_ids = dataframe["class_id"].to_numpy()
    return true_ids, predicted_ids, probabilities


## CELL 37 — Load and evaluate best Stage 1 checkpoint

**Repair record:** Completed the truncated validation output, retaining the best validation-loss checkpoint and recording all comparison metrics.


In [ ]:
stage1_model = tf.keras.models.load_model(STAGE1_PATH, compile=False)
y_validation, stage1_predictions, stage1_probabilities = get_predictions(stage1_model, validation_df, validation_ds)
stage1_accuracy = accuracy_score(y_validation, stage1_predictions)
stage1_macro_f1 = f1_score(y_validation, stage1_predictions, average="macro", zero_division=0)
stage1_balanced_accuracy = balanced_accuracy_score(y_validation, stage1_predictions)
print("Stage 1 validation accuracy:", stage1_accuracy)
print("Stage 1 validation macro-F1:", stage1_macro_f1)
print("Stage 1 validation balanced accuracy:", stage1_balanced_accuracy)


## CELL 38 — Start Stage 2 from the BEST Stage 1 model

Stage 2 starts from a separate copy of the best Stage 1 checkpoint. Backbone BatchNormalization stays frozen.

**Repair record:** Fixed the supplied alias model=stage1_model: reload an independent checkpoint before fine-tuning so Stage 1 remains selectable. Explicitly set both frozen and unfrozen flags.


In [ ]:
# Load a separate instance: Stage 2 must not mutate the Stage 1 candidate.
model = tf.keras.models.load_model(STAGE1_PATH, compile=False)
backbone = model.get_layer(BACKBONE_NAME)
backbone.trainable = True
FINE_TUNE_FRACTION = 0.25
fine_tune_from = int(len(backbone.layers) * (1.0 - FINE_TUNE_FRACTION))
for layer_index, layer in enumerate(backbone.layers):
    layer.trainable = (layer_index >= fine_tune_from
                       and not isinstance(layer, tf.keras.layers.BatchNormalization))
print("Trainable backbone layers:", sum(layer.trainable for layer in backbone.layers))
assert not any(layer.trainable for layer in backbone.layers
               if isinstance(layer, tf.keras.layers.BatchNormalization))


## CELL 39 — Recompile Stage 2

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)


## CELL 40 — Stage 2 callbacks

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
STAGE2_PATH = WORK_DIR / "stage2_best.keras"
stage2_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(str(STAGE2_PATH), monitor="val_loss", mode="min",
                                       save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=7,
                                     restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", mode="min", factor=0.3,
                                        patience=2, min_lr=1e-7, verbose=1),
]


## CELL 41 — Train Stage 2

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
history_stage2 = model.fit(train_ds, validation_data=validation_ds, epochs=35,
                           class_weight=class_weight, callbacks=stage2_callbacks, verbose=1)


## CELL 42 — Evaluate Stage 2

**Repair record:** Completed the truncated validation output using the best Stage 2 checkpoint.


In [ ]:
stage2_model = tf.keras.models.load_model(STAGE2_PATH, compile=False)
_, stage2_predictions, stage2_probabilities = get_predictions(stage2_model, validation_df, validation_ds)
stage2_accuracy = accuracy_score(y_validation, stage2_predictions)
stage2_macro_f1 = f1_score(y_validation, stage2_predictions, average="macro", zero_division=0)
stage2_balanced_accuracy = balanced_accuracy_score(y_validation, stage2_predictions)
print("Stage 2 validation accuracy:", stage2_accuracy)
print("Stage 2 validation macro-F1:", stage2_macro_f1)
print("Stage 2 validation balanced accuracy:", stage2_balanced_accuracy)


## CELL 43 — Compare both training stages

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
comparison_df = pd.DataFrame([
    {"model": "Stage 1 - Frozen EfficientNet", "validation_accuracy": stage1_accuracy,
     "validation_macro_f1": stage1_macro_f1, "validation_balanced_accuracy": stage1_balanced_accuracy},
    {"model": "Stage 2 - Fine-tuned EfficientNet", "validation_accuracy": stage2_accuracy,
     "validation_macro_f1": stage2_macro_f1, "validation_balanced_accuracy": stage2_balanced_accuracy},
])
display(comparison_df)


## CELL 44 — Automatically select the better model

Model selection uses validation macro-F1; the test set does not select a stage.

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.

**v5 update:** The default `reference_only_experimental` mode uses the existing two disease folders. A missing relevance model does not block reference-matching experimental class predictions. Current executed code and the v5 overview supersede earlier strict-only notes.


In [ ]:
if stage2_macro_f1 >= stage1_macro_f1:
    best_model = stage2_model
    chosen_stage = "stage2_fine_tuned"
    validation_probabilities = stage2_probabilities
else:
    best_model = stage1_model
    chosen_stage = "stage1_frozen"
    validation_probabilities = stage1_probabilities
print("SELECTED MODEL:", chosen_stage)
print("Selection uses validation macro-F1 only; ties retain the supplied Stage 2 preference.")


## CELL 44A — Configure labelled negative examples and audit separation

These optional negatives audit the old disease-reference distance only. They do not train or replace
input relevance. The current input-domain dataset is configured in CELL 44F. The legacy positive-only
experiment flag is ignored by the staged runtime and cannot enable diagnoses without relevance.

**v5 update:** The default `reference_only_experimental` mode uses the existing two disease folders. A missing relevance model does not block reference-matching experimental class predictions. Current executed code and the v5 overview supersede earlier strict-only notes.


In [ ]:
# Existing two-class data is sufficient for the default experimental workflow.
# Additional negative/relevance images are optional; no images or labels are invented.
INFERENCE_MODE = "reference_only_experimental"
# Optional strict mode requires a separately trained/calibrated relevance model:
# INFERENCE_MODE = "staged_relevance"
NEGATIVE_ROOT = Path("/content/bark_negative_examples")
OOD_K = 5
OOD_POSITIVE_QUANTILE = 0.975
MIN_VALIDATION_BARK_COVERAGE = 0.95
MAX_VALIDATION_NEGATIVE_PASS_RATE = 0.05
MIN_NEGATIVE_VALIDATION_IMAGES = 20
# Retained for compatibility with older packages; INFERENCE_MODE controls the v5 runtime.
ALLOW_POSITIVE_ONLY_DISEASE_RESULTS = False

negative_records = []
negative_excluded = []
known_bark_hashes = set(split_manifest["sha256"]) | set(conflicting_images["sha256"])
for split_name in ["validation", "test"]:
    folder = NEGATIVE_ROOT / split_name
    if not folder.exists():
        continue
    for path in sorted(folder.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        valid, error = validate_image(path)
        if not valid:
            negative_excluded.append({"path": str(path), "reason": "invalid_image", "detail": error})
            continue
        image_hash = sha256_file(path)
        if image_hash in known_bark_hashes:
            raise RuntimeError(f"Negative-labelled image duplicates a known bark image: {path}")
        negative_records.append({"path": str(path), "filename": path.name, "sha256": image_hash,
                                 "split": split_name, "category": path.parent.name, "class_id": 0})
negative_manifest = pd.DataFrame(negative_records,
    columns=["path", "filename", "sha256", "split", "category", "class_id"])
if not negative_manifest.empty:
    if (negative_manifest.groupby("sha256")["split"].nunique() > 1).any():
        raise RuntimeError("The same negative image occurs in validation and test. Fix the partitions first.")
    negative_manifest = negative_manifest.drop_duplicates("sha256").reset_index(drop=True)
negative_validation_df = negative_manifest[negative_manifest["split"] == "validation"].copy()
negative_test_df = negative_manifest[negative_manifest["split"] == "test"].copy()
print("Negative validation images:", len(negative_validation_df))
print("Negative held-out test images:", len(negative_test_df))
if negative_validation_df.empty:
    print("No optional distance-audit negatives supplied. Continue with the existing two disease folders.")
print("Reference-only filtering is experimental; unrelated-image rejection accuracy is unmeasured without negative tests.")


## CELL 44B — Define the shared reference-distance and decision runtime

Shared staged runtime: only calibrated input relevance can reject a decodable image. Disease-reference mismatch is uncertain; it cannot be overridden by high disease confidence.

**v5 update:** The default `reference_only_experimental` mode uses the existing two disease folders. A missing relevance model does not block reference-matching experimental class predictions. Current executed code and the v5 overview supersede earlier strict-only notes.


In [ ]:
rejection_runtime_code = r'''"""Training-reference OOD checks, not a cinnamon-species or healthy-bark detector."""
import hashlib
import json
from pathlib import Path
import numpy as np

CLASS_NAMES = ["RoughBark", "StripeCanker"]

def normalize_embeddings(values):
    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 2 or not len(values) or not np.isfinite(values).all():
        raise ValueError("Embeddings must be a nonempty finite matrix.")
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    if (norms <= 0).any():
        raise ValueError("A zero embedding cannot support a domain assessment.")
    return values / norms

def mean_knn_cosine_distance(query_embeddings, reference_embeddings, k=5, chunk_size=128):
    query = normalize_embeddings(query_embeddings)
    reference = normalize_embeddings(reference_embeddings)
    if query.shape[1] != reference.shape[1] or int(k) < 1 or int(chunk_size) < 1:
        raise ValueError("Invalid embedding dimensions, k or chunk size.")
    actual_k = min(int(k), len(reference))
    output = []
    for offset in range(0, len(query), int(chunk_size)):
        similarities = np.clip(query[offset:offset + int(chunk_size)] @ reference.T, -1.0, 1.0)
        distances = 1.0 - similarities
        neighbours = np.partition(distances, actual_k - 1, axis=1)[:, :actual_k]
        output.append(neighbours.mean(axis=1))
    return np.concatenate(output)

def domain_decisions_enabled(policy):
    # This gate only measures support for known diseases; it never establishes input relevance.
    return bool(np.isfinite(policy["distance_threshold"]))

def calibrate_relevance_policy(labels, scores, minimum_per_class=20,
                               min_positive_pass=0.90, max_negative_pass=0.05,
                               max_positive_reject=0.02, min_negative_reject=0.80):
    y, s = np.asarray(labels, dtype=int), np.asarray(scores, dtype=float)
    if y.ndim != 1 or s.shape != y.shape or not np.isfinite(s).all() or ((s < 0) | (s > 1)).any():
        raise ValueError("Invalid relevance validation labels/scores.")
    if not set(y.tolist()).issubset({0, 1}):
        raise ValueError("Relevance labels must be 0=not_relevant, 1=relevant_bark.")
    p, q = s[y == 1], s[y == 0]
    policy = {"schema_version": 1, "available": bool(len(s)),
        "class_names": ["not_relevant", "relevant_bark"], "positive_class_id": 1,
        "acceptance_enabled": False, "rejection_enabled": False,
        "accept_threshold": None, "reject_threshold": None,
        "positive_validation_images": int(len(p)), "negative_validation_images": int(len(q)),
        "minimum_validation_per_class": int(minimum_per_class),
        "min_positive_pass": min_positive_pass, "max_negative_pass": max_negative_pass,
        "max_positive_reject": max_positive_reject, "min_negative_reject": min_negative_reject,
        "calibration_source": "relevance_validation_only", "healthy_diagnosis_supported": False,
        "model_file": None, "model_sha256": None}
    rows=[]
    if not len(p) or not len(q):
        return policy, rows
    for t in np.unique(np.concatenate(([0.0, 1.0], s))):
        rows.append({"threshold": float(t), "positive_pass": float((p >= t).mean()),
                     "negative_pass": float((q >= t).mean()),
                     "positive_reject": float((p <= t).mean()),
                     "negative_reject": float((q <= t).mean())})
    if min(len(p),len(q)) < minimum_per_class:
        return policy, rows
    accepts=[r for r in rows if r["positive_pass"] >= min_positive_pass and r["negative_pass"] <= max_negative_pass]
    if accepts:
        selected=sorted(accepts,key=lambda r:(-r["positive_pass"],r["negative_pass"],r["threshold"]))[0]
        policy.update(acceptance_enabled=True,accept_threshold=selected["threshold"],
                      observed_positive_pass=selected["positive_pass"],observed_negative_pass=selected["negative_pass"])
    # A low threshold must not overlap the relevance acceptance region.
    upper=policy["accept_threshold"] if policy["acceptance_enabled"] else 1.0
    rejects=[r for r in rows if r["threshold"] < upper and r["positive_reject"] <= max_positive_reject
             and r["negative_reject"] >= min_negative_reject]
    if rejects:
        selected=sorted(rejects,key=lambda r:(-r["negative_reject"],r["positive_reject"],r["threshold"]))[0]
        policy.update(rejection_enabled=True,reject_threshold=selected["threshold"],
                      observed_positive_reject=selected["positive_reject"],observed_negative_reject=selected["negative_reject"])
    return policy,rows

def assess_relevance_score(score, policy):
    if score is None or not policy.get("available",False):
        return {"status":"uncertain","reason":"relevance_model_unavailable","score":None}
    score=float(score)
    if not np.isfinite(score) or not 0 <= score <= 1:
        raise ValueError("Invalid relevance score.")
    if policy.get("rejection_enabled",False) and score <= policy["reject_threshold"]:
        return {"status":"rejected","reason":"image_not_relevant","score":score}
    if policy.get("acceptance_enabled",False) and score >= policy["accept_threshold"]:
        return {"status":"passed","reason":"relevance_accepted","score":score}
    return {"status":"uncertain","reason":"relevance_unconfirmed_or_calibration_unmet","score":score}

def inference_mode(domain_policy):
    # Older packages stay strict unless the new experimental mode is explicitly recorded.
    mode = domain_policy.get("inference_mode", "staged_relevance")
    if mode not in {"reference_only_experimental", "staged_relevance"}:
        raise ValueError("Unknown bark inference mode.")
    return mode

def experimental_reference_fallback(domain_policy, relevance):
    # Never bypass an actual relevance rejection or an inconclusive trained model.
    return (inference_mode(domain_policy) == "reference_only_experimental"
            and relevance["reason"] == "relevance_model_unavailable"
            and relevance["score"] is None)

def _base_result(status, reason, distance, domain_policy, relevance):
    return {"status":status,"prediction":None,"candidate_class":None,
        "reason":reason,"confidence_score":None,"confidence_margin":None,"top_predictions":[],
        "domain_distance":None if distance is None else float(distance),
        "domain_distance_threshold":float(domain_policy["distance_threshold"]),
        "in_known_disease_reference":None if distance is None else bool(distance <= domain_policy["distance_threshold"]),
        "domain_gate_negative_targets_met":bool(domain_policy.get("negative_validation_targets_met",False)),
        "relevance_status":relevance["status"],"relevance_score":relevance["score"],
        "relevance_reason":relevance["reason"],"disease_classification_performed":False,
        "review_recommended":True,"healthy_detection_supported":False,"is_cinnamon":None,
        "assessment_mode":inference_mode(domain_policy),
        "input_relevance_verified":relevance["status"] == "passed",
        "unrelated_image_rejection_validated":False,
        "assessment_scope":("experimental_two_disease_reference_and_classifier"
            if experimental_reference_fallback(domain_policy,relevance)
            else "separate_relevance_then_known_disease_reference_and_classifier")}

def pre_disease_decision(distance, domain_policy, relevance):
    inference_mode(domain_policy)
    threshold=float(domain_policy["distance_threshold"])
    if not np.isfinite(threshold) or not 0 <= threshold <= 2:
        raise ValueError("Invalid disease-reference threshold.")
    if relevance["status"] == "rejected":
        return _base_result("rejected","image_not_relevant",None,domain_policy,relevance)
    if distance is not None:
        if not np.isfinite(distance) or not 0 <= distance <= 2:
            raise ValueError("Invalid disease-reference distance.")
        if distance > threshold:
            return _base_result("uncertain","outside_known_disease_bark_reference",distance,domain_policy,relevance)
    if relevance["status"] != "passed" and not experimental_reference_fallback(domain_policy,relevance):
        return _base_result("uncertain",relevance["reason"],distance,domain_policy,relevance)
    if distance is None:
        raise ValueError("Disease classification requires a disease-reference distance.")
    return None

def assess_prediction(probabilities, domain_distance, domain_policy, disease_policy, relevance_result=None):
    relevance=relevance_result or assess_relevance_score(None,{"available":False})
    stopped=pre_disease_decision(domain_distance,domain_policy,relevance)
    if stopped is not None:
        return stopped
    p=np.asarray(probabilities,dtype=np.float64)
    if (p.shape != (2,) or not np.isfinite(p).all() or (p < 0).any()
            or (p > 1).any() or not np.isclose(p.sum(),1,atol=1e-5,rtol=0)):
        raise ValueError("Invalid two-class softmax output.")
    ranking=np.argsort(-p,kind="stable")
    first,second=int(ranking[0]),int(ranking[1])
    confidence,margin=float(p[first]),float(p[first]-p[second])
    accepted=bool(disease_policy["acceptance_enabled"] and confidence >= disease_policy["confidence_threshold"]
                  and margin >= disease_policy["margin_threshold"])
    experimental = experimental_reference_fallback(domain_policy,relevance)
    status = ("experimental_candidate" if experimental else "disease_detected") if accepted else "uncertain"
    reason = (("reference_supported_experimental_candidate" if experimental else "supported_disease_candidate")
              if accepted else "insufficient_disease_evidence")
    result=_base_result(status,reason,domain_distance,domain_policy,relevance)
    result.update(prediction=CLASS_NAMES[first] if accepted else None,candidate_class=CLASS_NAMES[first],
        confidence_score=confidence,confidence_margin=margin,disease_classification_performed=True,
        review_recommended=experimental or not accepted,
        top_predictions=[{"class":CLASS_NAMES[int(i)],"confidence":float(p[i])} for i in ranking])
    return result

def predict_with_domain_gate(batch, embedding_predictor, disease_head_predictor, reference,
                             domain_policy, disease_policy, relevance_predictor=None,relevance_policy=None):
    policy=relevance_policy or {"available":False}
    score=None if relevance_predictor is None else float(np.asarray(relevance_predictor(batch)).reshape(-1)[0])
    relevance=assess_relevance_score(score,policy)
    if relevance["status"] == "rejected":
        return pre_disease_decision(None,domain_policy,relevance)
    # Distance can still explain lack of disease support when relevance is unavailable.
    embeddings=np.asarray(embedding_predictor(batch))
    if embeddings.ndim != 2 or len(embeddings) != 1:
        raise ValueError("Interactive prediction expects one image embedding.")
    distance=float(mean_knn_cosine_distance(embeddings,reference,k=domain_policy["k"])[0])
    stopped=pre_disease_decision(distance,domain_policy,relevance)
    if stopped is not None:
        return stopped
    probabilities=np.asarray(disease_head_predictor(embeddings))
    if probabilities.shape != (1,2):
        raise ValueError("Invalid disease-head output shape.")
    return assess_prediction(probabilities[0],distance,domain_policy,disease_policy,relevance)

def load_rejection_assets(directory, model_path):
    directory, model_path = Path(directory), Path(model_path)
    with open(directory / "ood_policy.json", encoding="utf-8") as file:
        policy = json.load(file)
    model_hash = hashlib.sha256(model_path.read_bytes()).hexdigest()
    expected = policy["model_sha256"].get(model_path.name)
    if expected is None or model_hash != expected:
        raise ValueError("OOD reference is not bound to this model artifact.")
    reference_path = directory / "ood_reference.npz"
    if hashlib.sha256(reference_path.read_bytes()).hexdigest() != policy["reference_sha256"]:
        raise ValueError("OOD reference checksum mismatch.")
    with np.load(reference_path, allow_pickle=False) as data:
        reference = np.asarray(data["embeddings"], dtype=np.float64)
        labels = np.asarray(data["class_ids"])
        image_hashes = np.asarray(data["image_sha256"])
    if (reference.shape != (policy["reference_images"], policy["embedding_dimension"])
            or labels.shape != (len(reference),) or set(labels.tolist()) != {0, 1}
            or image_hashes.shape != (len(reference),)):
        raise ValueError("Invalid OOD reference schema or class coverage.")
    normalize_embeddings(reference)
    return reference, policy
'''
exec(compile(rejection_runtime_code, "bark_rejection.py", "exec"), globals())


## CELL 44C — Build references from unaugmented training images only

No validation or test image is placed in the reference bank. This is a new post-training check on the selected model; it does not alter EfficientNet weights or either disease label.

**v5 update:** The default `reference_only_experimental` mode uses the existing two disease folders. A missing relevance model does not block reference-matching experimental class predictions. Current executed code and the v5 overview supersede earlier strict-only notes.


In [ ]:
embedding_model = tf.keras.Model(best_model.inputs, best_model.get_layer("global_average_pooling").output)

def collect_embeddings(extractor, dataframe):
    # make_dataset(..., training=False) preserves manifest order and never augments or shuffles.
    values = extractor.predict(make_dataset(dataframe, training=False), verbose=0)
    return normalize_embeddings(values)

train_embeddings_normalized = collect_embeddings(embedding_model, train_df)
validation_embeddings_normalized = collect_embeddings(embedding_model, validation_df)
train_embedding_labels = train_df["class_id"].to_numpy(dtype=np.int32)
validation_ood_distances = mean_knn_cosine_distance(
    validation_embeddings_normalized, train_embeddings_normalized, k=OOD_K)
assert len(train_embeddings_normalized) == len(train_df)
print("Training reference embeddings:", train_embeddings_normalized.shape)
print("Validation distance min/median/max:",
      float(validation_ood_distances.min()), float(np.median(validation_ood_distances)),
      float(validation_ood_distances.max()))


## CELL 44D — Calibrate the gate using validation data

Calibrate known-disease support on validation data, with optional negative-distance diagnostics. This is not a relevance classifier and its threshold is never loosened for a single inspected photo.

**v5 update:** The default `reference_only_experimental` mode uses the existing two disease folders. A missing relevance model does not block reference-matching experimental class predictions. Current executed code and the v5 overview supersede earlier strict-only notes.


In [ ]:
negative_validation_distances = np.array([], dtype=np.float64)
if not negative_validation_df.empty:
    negative_validation_embeddings = collect_embeddings(embedding_model, negative_validation_df)
    negative_validation_distances = mean_knn_cosine_distance(
        negative_validation_embeddings, train_embeddings_normalized, k=OOD_K)

positive_only_threshold = float(np.quantile(validation_ood_distances, OOD_POSITIVE_QUANTILE))
gate_calibration_rows = []
negative_targets_met = False
OOD_DISTANCE_THRESHOLD = positive_only_threshold
if len(negative_validation_distances):
    candidates = np.unique(np.concatenate([validation_ood_distances, negative_validation_distances]))
    for threshold in candidates:
        gate_calibration_rows.append({
            "distance_threshold": float(threshold),
            "bark_coverage": float((validation_ood_distances <= threshold).mean()),
            "negative_pass_rate": float((negative_validation_distances <= threshold).mean()),
        })
gate_calibration_df = pd.DataFrame(gate_calibration_rows,
    columns=["distance_threshold", "bark_coverage", "negative_pass_rate"])
if len(negative_validation_distances) >= MIN_NEGATIVE_VALIDATION_IMAGES:
    eligible_gate_policies = gate_calibration_df[
        (gate_calibration_df["bark_coverage"] >= MIN_VALIDATION_BARK_COVERAGE)
        & (gate_calibration_df["negative_pass_rate"] <= MAX_VALIDATION_NEGATIVE_PASS_RATE)]
    if not eligible_gate_policies.empty:
        selected_gate = eligible_gate_policies.sort_values(
            ["bark_coverage", "negative_pass_rate", "distance_threshold"],
            ascending=[False, True, True], kind="stable").iloc[0]
        OOD_DISTANCE_THRESHOLD = float(selected_gate["distance_threshold"])
        negative_targets_met = True

validation_in_domain = validation_ood_distances <= OOD_DISTANCE_THRESHOLD
OOD_POLICY = {
    "schema_version": 3, "inference_mode": INFERENCE_MODE, "purpose": "known_disease_support_only", "outside_reference_status": "uncertain", "method": "mean_knn_cosine_distance", "k": OOD_K,
    "distance_threshold": OOD_DISTANCE_THRESHOLD,
    "reference_source": "unaugmented_training_images_only",
    "embedding_layer": "global_average_pooling",
    "embedding_dimension": int(train_embeddings_normalized.shape[1]),
    "reference_images": int(len(train_embeddings_normalized)),
    "calibration_source": "positive_and_negative_validation" if negative_targets_met else "positive_validation_quantile_experimental",
    "positive_validation_quantile": OOD_POSITIVE_QUANTILE,
    "validation_bark_coverage": float(validation_in_domain.mean()),
    "negative_validation_images": int(len(negative_validation_distances)),
    "negative_validation_pass_rate": (float((negative_validation_distances <= OOD_DISTANCE_THRESHOLD).mean())
                                      if len(negative_validation_distances) else None),
    "minimum_negative_validation_images": MIN_NEGATIVE_VALIDATION_IMAGES,
    "target_bark_coverage": MIN_VALIDATION_BARK_COVERAGE,
    "target_negative_pass_rate": MAX_VALIDATION_NEGATIVE_PASS_RATE,
    "negative_validation_targets_met": bool(negative_targets_met),
    "allow_positive_only_disease_results": bool(ALLOW_POSITIVE_ONLY_DISEASE_RESULTS),
    "species_verification_supported": False, "healthy_detection_supported": False,
    "limitation": "Observed validation targets are not a guarantee for unseen objects, other bark, or healthy cinnamon bark.",
}
print(json.dumps(OOD_POLICY, indent=2))
print("This threshold measures similarity to known diseases. In reference-only mode, passing images can receive experimental class predictions.")


## CELL 44E — Define model analysis and the gated prediction policy

Defines raw joint analysis for disease metrics/parity. The complete interactive inference function is connected after relevance calibration in CELL 44I.

**v5 update:** The default `reference_only_experimental` mode uses the existing two disease folders. A missing relevance model does not block reference-matching experimental class predictions. Current executed code and the v5 overview supersede earlier strict-only notes.


In [ ]:
def build_bark_analysis_model(trained_model):
    # Joint output is retained for raw classifier evaluation and serialization diagnostics only.
    return tf.keras.Model(trained_model.inputs, [
        trained_model.output, trained_model.get_layer("global_average_pooling").output])

analysis_model = build_bark_analysis_model(best_model)

def current_disease_policy():
    return {"acceptance_enabled": bool(ACCEPTANCE_ENABLED),
            "confidence_threshold": CONFIDENCE_THRESHOLD, "margin_threshold": MARGIN_THRESHOLD}

def analyse_bark_batch(batch, trained_model=best_model):
    feature_model = (embedding_model if trained_model is best_model else tf.keras.Model(
        trained_model.inputs, trained_model.get_layer("global_average_pooling").output))
    dropout = trained_model.get_layer("classification_dropout")
    disease_output = trained_model.get_layer("bark_predictions")
    def run_features(images):
        return np.asarray(feature_model(images, training=False))
    def run_disease_head(features):
        return np.asarray(disease_output(dropout(tf.convert_to_tensor(features), training=False)))
    return predict_with_domain_gate(batch, run_features, run_disease_head,
        train_embeddings_normalized, OOD_POLICY, current_disease_policy())


## CELL 44F — Prepare a separate labelled relevance dataset (optional)

Supply `/content/bark_relevance_dataset/{train,validation,test}/{not_relevant,relevant_bark}/`.
`relevant_bark` needs confirmed cinnamon bark/stem images across views/cameras/backgrounds; healthy-looking or unlabelled bark can be used here without any disease label.
`not_relevant` needs other-tree bark, leaves and diverse unrelated photos. Split by independent trees/source/capture sessions before upload; hash checks do not prove group independence.
Do not copy this newly tested image into validation and then claim an independent test. No new images or labels are fabricated. With no dataset, training is skipped and relevance remains unavailable.

**v5 update:** The default `reference_only_experimental` mode uses the existing two disease folders. A missing relevance model does not block reference-matching experimental class predictions. Current executed code and the v5 overview supersede earlier strict-only notes.


In [ ]:
RELEVANCE_ROOT=Path("/content/bark_relevance_dataset")
RELEVANCE_CLASS_NAMES=["not_relevant","relevant_bark"]
RELEVANCE_TRAIN_EPOCHS=15
RELEVANCE_FINE_TUNE_EPOCHS=10
MIN_RELEVANCE_TRAIN_PER_CLASS=20
MIN_RELEVANCE_VALIDATION_PER_CLASS=20
relevance_records=[]
relevance_bad=[]
disease_split_by_hash=dict(zip(split_manifest["sha256"],split_manifest["split"]))
for split in ["train","validation","test"]:
    for class_id,class_name in enumerate(RELEVANCE_CLASS_NAMES):
        folder=RELEVANCE_ROOT/split/class_name
        if not folder.exists():
            continue
        for image_path in sorted(folder.rglob("*")):
            if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            valid,error=validate_image(image_path)
            if not valid:
                relevance_bad.append({"path":str(image_path),"error":error});continue
            digest=sha256_file(image_path)
            if digest in disease_split_by_hash:
                if class_id == 0 or split != disease_split_by_hash[digest]:
                    raise RuntimeError(f"Relevance labels/splits conflict with known disease images: {image_path}")
            relevance_records.append({"path":str(image_path),"class_id":class_id,"class_name":class_name,
                                      "split":split,"sha256":digest,"filename":image_path.name})
relevance_manifest=pd.DataFrame(relevance_records,columns=["path","class_id","class_name","split","sha256","filename"])
if not relevance_manifest.empty:
    if (relevance_manifest.groupby("sha256")["split"].nunique()>1).any():
        raise RuntimeError("Identical relevance images cross splits. Keep each source/capture group in one partition.")
    if (relevance_manifest.groupby("sha256")["class_id"].nunique()>1).any():
        raise RuntimeError("Identical images have opposing relevance labels.")
    relevance_manifest=relevance_manifest.drop_duplicates("sha256").reset_index(drop=True)
relevance_parts={split:relevance_manifest[relevance_manifest["split"]==split].copy()
                 for split in ["train","validation","test"]}
relevance_training_available=not relevance_manifest.empty
if relevance_training_available:
    for split,minimum in [("train",MIN_RELEVANCE_TRAIN_PER_CLASS),("validation",MIN_RELEVANCE_VALIDATION_PER_CLASS),("test",1)]:
        counts=relevance_parts[split]["class_id"].value_counts().reindex([0,1],fill_value=0)
        if (counts<minimum).any():
            raise RuntimeError(f"Relevance {split} requires >= {minimum} images in each class; counts: {counts.to_dict()}")
    display(relevance_manifest.groupby(["split","class_name"]).size())
else:
    print("Optional relevance training skipped: no additional dataset supplied.")
    print("No extra dataset is required in reference_only_experimental mode. Reference-matching images proceed to disease classification.")


## CELL 44G — Train and select an independent MobileNetV3Small relevance model (optional)

This optional binary relevance task is separate from the two disease labels. Input remains RGB float32 `[0,255]`; MobileNetV3Small includes its own preprocessing. A score means model-estimated relevance, not disease or healthy status. The disease model is not modified by this training. Model and threshold selection never use relevance test images.

**v5 update:** The default `reference_only_experimental` mode uses the existing two disease folders. A missing relevance model does not block reference-matching experimental class predictions. Current executed code and the v5 overview supersede earlier strict-only notes.


In [ ]:
from sklearn.metrics import roc_auc_score
relevance_model=None
relevance_history=pd.DataFrame()
relevance_validation_scores=np.array([],dtype=float)
relevance_training_summary={"trained":False,"architecture":"MobileNetV3Small","selection":"validation_roc_auc"}
if relevance_training_available:
    rtrain=make_dataset(relevance_parts["train"],training=True)
    rval=make_dataset(relevance_parts["validation"],training=False)
    rbase=tf.keras.applications.MobileNetV3Small(include_top=False,weights="imagenet",
        input_shape=(224,224,3),include_preprocessing=True,pooling="avg")
    rbase.trainable=False
    rinput=tf.keras.Input((224,224,3),dtype="float32",name="relevance_image")
    rx=rbase(rinput,training=False)
    rx=tf.keras.layers.Dropout(0.2,name="relevance_dropout")(rx)
    rout=tf.keras.layers.Dense(1,activation="sigmoid",dtype="float32",name="relevance_score")(rx)
    rmodel=tf.keras.Model(rinput,rout,name="cinnamon_bark_relevance")
    rbase_name=rbase.name
    rmodel.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss="binary_crossentropy",metrics=["accuracy"])
    rweights=compute_class_weight(class_weight="balanced",classes=np.array([0,1]),y=relevance_parts["train"]["class_id"])
    rclass_weights={int(i):float(w) for i,w in enumerate(rweights)}
    rhead_path=WORK_DIR/"relevance_head_best.keras"
    rhead_history=rmodel.fit(rtrain,validation_data=rval,epochs=RELEVANCE_TRAIN_EPOCHS,
        class_weight=rclass_weights,callbacks=[
            tf.keras.callbacks.ModelCheckpoint(str(rhead_path),monitor="val_loss",save_best_only=True),
            tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=4,restore_best_weights=True)],verbose=1)
    rhead_model=tf.keras.models.load_model(rhead_path,compile=False)
    rhead_scores=rhead_model.predict(rval,verbose=0).reshape(-1)
    rlabels=relevance_parts["validation"]["class_id"].to_numpy()
    rhead_auc=float(roc_auc_score(rlabels,rhead_scores))
    # Independent copy, so fine-tuning cannot mutate the head-only candidate or the disease model.
    rfine=tf.keras.models.load_model(rhead_path,compile=False)
    rfine_base=rfine.get_layer(rbase_name)
    rfine_base.trainable=True
    for index,layer in enumerate(rfine_base.layers):
        layer.trainable=index>=int(len(rfine_base.layers)*0.75) and not isinstance(layer,tf.keras.layers.BatchNormalization)
    rfine.compile(optimizer=tf.keras.optimizers.Adam(1e-5),loss="binary_crossentropy",metrics=["accuracy"])
    rfine_path=WORK_DIR/"relevance_finetuned_best.keras"
    rfine_history=rfine.fit(rtrain,validation_data=rval,epochs=RELEVANCE_FINE_TUNE_EPOCHS,
        class_weight=rclass_weights,callbacks=[
            tf.keras.callbacks.ModelCheckpoint(str(rfine_path),monitor="val_loss",save_best_only=True),
            tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=4,restore_best_weights=True)],verbose=1)
    rfine_model=tf.keras.models.load_model(rfine_path,compile=False)
    rfine_scores=rfine_model.predict(rval,verbose=0).reshape(-1)
    rfine_auc=float(roc_auc_score(rlabels,rfine_scores))
    if rfine_auc>rhead_auc:
        relevance_model,relevance_validation_scores=rfine_model,rfine_scores
        relevance_stage="fine_tuned"
    else:
        relevance_model,relevance_validation_scores=rhead_model,rhead_scores
        relevance_stage="frozen_backbone"
    relevance_history=pd.concat([pd.DataFrame(rhead_history.history).assign(stage="head"),
                                 pd.DataFrame(rfine_history.history).assign(stage="fine_tune")],ignore_index=True)
    relevance_training_summary.update(trained=True,selected_stage=relevance_stage,
        head_validation_auc=rhead_auc,fine_tune_validation_auc=rfine_auc)
    print(relevance_training_summary)


## CELL 44H — Calibrate accept, reject and uncertain relevance regions (optional)

Accept thresholds target observed relevant retention >=90% and negative pass <=5%; reject thresholds separately target relevant false rejection <=2% and negative rejection >=80%. Targets are configurable experiments, not guarantees. At least 20 validation images per class are an execution minimum; larger independent sets and difficult negatives are needed for credible field validation. Unmet goals disable the corresponding decision rather than invent a threshold.

**v5 update:** The default `reference_only_experimental` mode uses the existing two disease folders. A missing relevance model does not block reference-matching experimental class predictions. Current executed code and the v5 overview supersede earlier strict-only notes.


In [ ]:
relevance_labels=relevance_parts["validation"]["class_id"].to_numpy(dtype=int)
RELEVANCE_POLICY,relevance_threshold_rows=calibrate_relevance_policy(
    relevance_labels,relevance_validation_scores,minimum_per_class=MIN_RELEVANCE_VALIDATION_PER_CLASS)
RELEVANCE_POLICY["architecture"]="MobileNetV3Small"
RELEVANCE_POLICY["group_independence_verified"]=False
print(json.dumps(RELEVANCE_POLICY,indent=2))
print("Only validated low relevance scores can reject an otherwise decodable image.")
print("Unknown relevance and unsupported disease appearance remain uncertain; neither is a healthy diagnosis.")


## CELL 44I — Connect relevance before the disease-reference check

Final order: calibrated relevance → known-disease reference → disease confidence. A valid bark image outside the two-disease reference returns **uncertain**, not rejected. Without a trained relevance gate, no image is promoted to an accepted disease diagnosis merely because it resembles a training photo.

**v5 update:** The default `reference_only_experimental` mode uses the existing two disease folders. A missing relevance model does not block reference-matching experimental class predictions. Current executed code and the v5 overview supersede earlier strict-only notes.


In [ ]:
# Also applies the fix to an already-trained Colab session: rerun 44B, then this cell.
INFERENCE_MODE = globals().get("INFERENCE_MODE", "reference_only_experimental")
OOD_POLICY["schema_version"] = 3
OOD_POLICY["inference_mode"] = INFERENCE_MODE
inference_mode(OOD_POLICY)  # Validate the selected mode before prediction/export.
print("Inference mode:", INFERENCE_MODE)
if INFERENCE_MODE == "reference_only_experimental" and relevance_model is None:
    print("Using the existing two disease classes. No separate relevance dataset is required.")
    print("Reference + disease-score passes yield experimental_candidate, with relevance unverified.")

def relevance_for_batch(batch,model=None,policy=None):
    selected=relevance_model if model is None else model
    selected_policy=RELEVANCE_POLICY if policy is None else policy
    score=None if selected is None else float(np.asarray(selected(batch,training=False)).reshape(-1)[0])
    return assess_relevance_score(score,selected_policy)

def analyse_bark_batch(batch,trained_model=best_model):
    feature_model=embedding_model if trained_model is best_model else tf.keras.Model(
        trained_model.inputs,trained_model.get_layer("global_average_pooling").output)
    def features(images): return np.asarray(feature_model(images,training=False))
    def head(values):
        dropped=trained_model.get_layer("classification_dropout")(tf.convert_to_tensor(values),training=False)
        return np.asarray(trained_model.get_layer("bark_predictions")(dropped))
    rpredict=None if relevance_model is None else lambda images: np.asarray(relevance_model(images,training=False))
    return predict_with_domain_gate(batch,features,head,train_embeddings_normalized,OOD_POLICY,
                                   current_disease_policy(),rpredict,RELEVANCE_POLICY)


## CELL 45 — Plot Stage 1 accuracy

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(history_stage1.history["accuracy"], label="Training accuracy")
plt.plot(history_stage1.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Stage 1 Accuracy")
plt.legend()
plt.show()


## CELL 46 — Plot Stage 1 loss

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(history_stage1.history["loss"], label="Training loss")
plt.plot(history_stage1.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Stage 1 Loss")
plt.legend()
plt.show()


## CELL 47 — Plot Stage 2 accuracy

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(history_stage2.history["accuracy"], label="Training accuracy")
plt.plot(history_stage2.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Stage 2 Accuracy")
plt.legend()
plt.show()


## CELL 48 — Plot Stage 2 loss

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(history_stage2.history["loss"], label="Training loss")
plt.plot(history_stage2.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Stage 2 Loss")
plt.legend()
plt.show()


## CELL 49 — Combine and save training history

**Repair record:** Closed the truncated training-history export call.


In [ ]:
history_stage1_df = pd.DataFrame(history_stage1.history)
history_stage1_df["stage"] = "stage1"
history_stage1_df["epoch"] = np.arange(1, len(history_stage1_df) + 1)
history_stage2_df = pd.DataFrame(history_stage2.history)
history_stage2_df["stage"] = "stage2"
history_stage2_df["epoch"] = np.arange(1, len(history_stage2_df) + 1)
training_history = pd.concat([history_stage1_df, history_stage2_df], ignore_index=True)
training_history.to_csv(WORK_DIR / "training_history.csv", index=False)
display(training_history.tail())


## CELL 50 — Calculate validation confidence and margin

Confidence and margin support an uncertain state between the two disease labels; they do not establish healthy detection.

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
validation_predictions = np.argmax(validation_probabilities, axis=1)
sorted_validation_scores = np.sort(validation_probabilities, axis=1)
validation_confidence = sorted_validation_scores[:, -1]
validation_margin = sorted_validation_scores[:, -1] - sorted_validation_scores[:, -2]
validation_correct = validation_predictions == y_validation


## CELL 51 — Search validation thresholds

**Repair record:** Completed the truncated threshold search, accepted-accuracy/coverage calculations and missing threshold_df definition.

**Input-rejection update:** Include validation reference acceptance when choosing disease-score thresholds; keep test data out of calibration.


In [ ]:
threshold_results = []
for confidence_threshold in np.arange(0.50, 0.96, 0.02):
    for margin_threshold in np.arange(0.00, 0.51, 0.02):
        accepted = ((validation_confidence >= confidence_threshold)
                    & (validation_margin >= margin_threshold) & validation_in_domain)
        accepted_count = int(accepted.sum())
        if accepted_count == 0:
            continue
        threshold_results.append({
            "confidence_threshold": float(confidence_threshold),
            "margin_threshold": float(margin_threshold),
            "accepted_count": accepted_count,
            "coverage": float(accepted.mean()),
            "accepted_accuracy": float(validation_correct[accepted].mean()),
        })
threshold_df = pd.DataFrame(threshold_results)
if threshold_df.empty:
    raise RuntimeError("No validation examples met even the lowest score thresholds. Inspect model outputs.")
display(threshold_df.sort_values(["coverage", "accepted_accuracy"], ascending=False).head(15))


## CELL 52 — Automatically choose uncertainty threshold

Targets are 90% accepted validation accuracy and 60% validation coverage. They are goals, not guaranteed performance.

**Repair record:** Completed threshold selection and scalar definitions. Documented a conservative missing-target fallback: disable acceptance instead of claiming unmet 90%/60% goals.

**Input-rejection update:** Distinguish disease-score targets from the separate negative-calibrated reference gate.

**Staged relevance update:** Require calibrated relevance acceptance alongside disease-score validation; the disease-distance gate is not an input classifier.


In [ ]:
TARGET_ACCEPTED_ACCURACY = 0.90
MIN_VALIDATION_COVERAGE = 0.60
eligible_thresholds = threshold_df[
    (threshold_df["accepted_accuracy"] >= TARGET_ACCEPTED_ACCURACY)
    & (threshold_df["coverage"] >= MIN_VALIDATION_COVERAGE)
].copy()
ACCEPTANCE_ENABLED = not eligible_thresholds.empty
if ACCEPTANCE_ENABLED:
    selected_threshold = eligible_thresholds.sort_values(
        ["coverage", "accepted_accuracy", "confidence_threshold", "margin_threshold"],
        ascending=[False, False, True, True], kind="stable",
    ).iloc[0]
else:
    selected_threshold = threshold_df.sort_values(
        ["accepted_accuracy", "coverage"], ascending=False, kind="stable",
    ).iloc[0]
    print("Validation targets were not met. Automatic accepted disease results are disabled; results remain uncertain.")
CONFIDENCE_THRESHOLD = float(selected_threshold["confidence_threshold"])
MARGIN_THRESHOLD = float(selected_threshold["margin_threshold"])
print("Confidence threshold:", CONFIDENCE_THRESHOLD)
print("Margin threshold:", MARGIN_THRESHOLD)
print("Disease-score validation targets met:", ACCEPTANCE_ENABLED)
print("Experimental class results enabled:", bool(ACCEPTANCE_ENABLED and INFERENCE_MODE == "reference_only_experimental"))
print("Relevance-verified disease results enabled:", bool(ACCEPTANCE_ENABLED and RELEVANCE_POLICY["acceptance_enabled"]))
display(selected_threshold.to_frame("selected_validation_policy"))


## CELL 53 — FINAL held-out test evaluation

This is the first test-performance measurement. Do not use these results to tune the model or thresholds.

**Repair record:** Completed the truncated held-out evaluation and per-class classification report; no healthy-class metrics are invented.


In [ ]:
y_test, test_predictions, test_probabilities = get_predictions(best_model, test_df, test_ds)
test_accuracy = accuracy_score(y_test, test_predictions)
test_macro_f1 = f1_score(y_test, test_predictions, average="macro", zero_division=0)
test_balanced_accuracy = balanced_accuracy_score(y_test, test_predictions)
print("=" * 70)
print("TEST ACCURACY:", round(test_accuracy, 4))
print("TEST MACRO-F1:", round(test_macro_f1, 4))
print("TEST BALANCED ACCURACY:", round(test_balanced_accuracy, 4))
print(classification_report(y_test, test_predictions, labels=[0, 1], target_names=CLASS_NAMES, zero_division=0))
print("These are two-disease metrics; healthy/unhealthy accuracy is not established.")


## CELL 53A — Evaluate the frozen rejection policy on held-out data

Evaluate frozen staged decisions and both held-out test populations. No model/reference/threshold is changed using these results.

**v5 update:** The default `reference_only_experimental` mode uses the existing two disease folders. A missing relevance model does not block reference-matching experimental class predictions. Current executed code and the v5 overview supersede earlier strict-only notes.


In [ ]:
# No model, reference, or threshold is changed in this section.
test_embeddings_normalized = collect_embeddings(embedding_model, test_df)
test_ood_distances = mean_knn_cosine_distance(test_embeddings_normalized, train_embeddings_normalized, k=OOD_K)
test_relevance_scores=(np.full(len(test_df),np.nan) if relevance_model is None else
    relevance_model.predict(make_dataset(test_df,training=False),verbose=0).reshape(-1))
test_gate_results=[assess_prediction(p,float(d),OOD_POLICY,current_disease_policy(),
    assess_relevance_score(None if relevance_model is None else float(r),RELEVANCE_POLICY))
    for p,d,r in zip(test_probabilities,test_ood_distances,test_relevance_scores)]
negative_test_results = pd.DataFrame(columns=["path", "category", "domain_distance", "status", "prediction"])
if not negative_test_df.empty:
    negative_test_probabilities, negative_test_embeddings = [np.asarray(value) for value in
        analysis_model.predict(make_dataset(negative_test_df, training=False), verbose=0)]
    negative_test_distances = mean_knn_cosine_distance(
        negative_test_embeddings, train_embeddings_normalized, k=OOD_K)
    negative_relevance_scores=(np.full(len(negative_test_df),np.nan) if relevance_model is None else
        relevance_model.predict(make_dataset(negative_test_df,training=False),verbose=0).reshape(-1))
    negative_rows = []
    for (_, row), probabilities, distance, rscore in zip(
            negative_test_df.iterrows(), negative_test_probabilities, negative_test_distances, negative_relevance_scores):
        result = assess_prediction(probabilities,float(distance),OOD_POLICY,current_disease_policy(),
            assess_relevance_score(None if relevance_model is None else float(rscore),RELEVANCE_POLICY))
        negative_rows.append({"path": row["path"], "category": row["category"], **result})
    negative_test_results = pd.DataFrame(negative_rows)
OOD_TEST_METRICS = {
    "inference_mode": INFERENCE_MODE,
    "known_disease_test_images": int(len(test_df)),
    "known_disease_reference_coverage": float((test_ood_distances <= OOD_DISTANCE_THRESHOLD).mean()),
    "negative_test_images": int(len(negative_test_results)),
    "negative_test_rejection_rate": (float((negative_test_results["status"] == "rejected").mean())
                                     if len(negative_test_results) else None),
    "negative_test_disease_result_rate": (float((negative_test_results["status"] == "disease_detected").mean())
                                          if len(negative_test_results) else None),
    "negative_test_experimental_candidate_rate": (float((negative_test_results["status"] == "experimental_candidate").mean())
                                                   if len(negative_test_results) else None),
    "negative_test_reference_pass_rate": (float((negative_test_results["domain_distance"] <= OOD_DISTANCE_THRESHOLD).mean())
                                           if len(negative_test_results) else None),
    "species_verification_supported": False,
}
print(json.dumps(OOD_TEST_METRICS, indent=2))
if negative_test_results.empty:
    print("No held-out negatives supplied. Unrelated-image rejection performance is unmeasured, not 100%.")

relevance_test_metrics={"images":0,"positive_images":0,"negative_images":0,"positive_false_reject_rate":None,
                        "negative_false_pass_rate":None,"uncertain_rate":None,"roc_auc":None}
if relevance_model is not None:
    rtest=relevance_parts["test"]
    rscores=relevance_model.predict(make_dataset(rtest,training=False),verbose=0).reshape(-1)
    rtruth=rtest["class_id"].to_numpy()
    rstatus=np.array([assess_relevance_score(float(score),RELEVANCE_POLICY)["status"] for score in rscores])
    relevance_test_metrics={"images":int(len(rtest)),"positive_images":int((rtruth==1).sum()),
        "negative_images":int((rtruth==0).sum()),"positive_false_reject_rate":float((rstatus[rtruth==1]=="rejected").mean()),
        "negative_false_pass_rate":float((rstatus[rtruth==0]=="passed").mean()),
        "uncertain_rate":float((rstatus=="uncertain").mean()),"roc_auc":float(roc_auc_score(rtruth,rscores))}
    relevance_test_predictions=rtest.assign(relevance_score=rscores,relevance_status=rstatus)
else:
    relevance_test_predictions=pd.DataFrame(columns=["path","class_id","relevance_score","relevance_status"])
print("Independent relevance test:",json.dumps(relevance_test_metrics,indent=2))


## CELL 54 — Test confusion matrix

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
confusion = confusion_matrix(y_test, test_predictions, labels=[0, 1])
display_object = ConfusionMatrixDisplay(confusion_matrix=confusion, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(7, 7))
display_object.plot(ax=ax, values_format="d")
plt.title("Cinnamon Bark Disease Test Confusion Matrix")
plt.show()


## CELL 55 — Apply uncertainty policy to test results

**Repair record:** Completed truncated accepted-test metrics and handled zero accepted examples without NaN or division errors.

**Input-rejection update:** Measure the full reference/negative-validation/confidence policy rather than softmax confidence alone.

**v5 update:** The default `reference_only_experimental` mode uses the existing two disease folders. A missing relevance model does not block reference-matching experimental class predictions. Current executed code and the v5 overview supersede earlier strict-only notes.


In [ ]:
sorted_test_scores = np.sort(test_probabilities, axis=1)
test_confidence = sorted_test_scores[:, -1]
test_margin = sorted_test_scores[:, -1] - sorted_test_scores[:, -2]
test_accepted = np.array([r["status"] == "disease_detected" for r in test_gate_results], dtype=bool)
test_coverage = float(test_accepted.mean())
accepted_test_accuracy = (float((test_predictions[test_accepted] == y_test[test_accepted]).mean())
                          if test_accepted.any() else None)
print("Accepted test disease results:", int(test_accepted.sum()), "/", len(test_df))
print("Rejected reference images:", sum(r["status"] == "rejected" for r in test_gate_results))
print("Test assessment coverage:", test_coverage)
print("Accepted test accuracy:", accepted_test_accuracy)

test_experimental = np.array([r["status"] == "experimental_candidate" for r in test_gate_results], dtype=bool)
experimental_test_coverage = float(test_experimental.mean())
experimental_test_accuracy = (float((test_predictions[test_experimental] == y_test[test_experimental]).mean())
                              if test_experimental.any() else None)
print("Experimental test candidates:", int(test_experimental.sum()), "/", len(test_df))
print("Experimental candidate coverage:", experimental_test_coverage)
print("Experimental candidate accuracy on held-out known diseases:", experimental_test_accuracy)
print("This is separate from relevance-verified disease coverage and unrelated-image rejection accuracy.")


## CELL 56 — Build detailed test-prediction table

**Repair record:** Completed the truncated result columns used by error analysis and final CSV export.

**Input-rejection update:** Save rejected, uncertain and accepted statuses plus the reference distance and decision reason.

**v5 update:** The default `reference_only_experimental` mode uses the existing two disease folders. A missing relevance model does not block reference-matching experimental class predictions. Current executed code and the v5 overview supersede earlier strict-only notes.


In [ ]:
test_results = test_df.reset_index(drop=True).copy()
test_results["predicted_class_id"] = test_predictions
test_results["predicted_class"] = [CLASS_NAMES[class_id] for class_id in test_predictions]
test_results["rough_bark_score"] = test_probabilities[:, 0]
test_results["stripe_canker_score"] = test_probabilities[:, 1]
test_results["confidence"] = test_confidence
test_results["margin"] = test_margin
test_results["correct"] = test_predictions == y_test
test_results["accepted"] = test_accepted
test_results["experimental_candidate"] = test_experimental
test_results["input_relevance_verified"] = [r["input_relevance_verified"] for r in test_gate_results]
test_results["assessment_mode"] = INFERENCE_MODE
test_results["status"] = [r["status"] for r in test_gate_results]
test_results["domain_distance"] = test_ood_distances
test_results["reason"] = [r["reason"] for r in test_gate_results]
display(test_results.head())


## CELL 57 — Show misclassifications

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
mistakes = test_results[~test_results["correct"]].copy()
print("Misclassified test images:", len(mistakes))
display(mistakes[["filename", "class_name", "predicted_class", "confidence", "margin"]])


## CELL 58 — Display misclassified images

**Repair record:** Completed the truncated error-image display and empty-mistakes handling.


In [ ]:
NUMBER_TO_SHOW = min(12, len(mistakes))
if NUMBER_TO_SHOW > 0:
    number_of_rows = math.ceil(NUMBER_TO_SHOW / 4)
    fig, axes = plt.subplots(number_of_rows, 4, figsize=(16, 4 * number_of_rows), squeeze=False)
    axes = axes.reshape(-1)
    for index, (_, row) in enumerate(mistakes.head(NUMBER_TO_SHOW).iterrows()):
        with Image.open(row["path"]) as image:
            axes[index].imshow(ImageOps.exif_transpose(image).convert("RGB"))
        axes[index].set_title(f"True: {row['class_name']}\nPredicted: {row['predicted_class']}")
    for axis in axes:
        axis.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No misclassified images in this test partition.")


## CELL 59 — Single image preprocessing helper

**Repair record:** Completed the truncated return tuple while retaining the same preprocessing as training.


In [ ]:
def preprocess_single_image(image_path):
    with Image.open(image_path) as image:
        image = ImageOps.exif_transpose(image).convert("RGB")
        original_image = image.copy()
        image = image.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
        array = np.asarray(image, dtype=np.float32)
    batch = np.expand_dims(array, axis=0)
    return original_image, batch


## CELL 60 — Application-style prediction function

Uses the final staged function from CELL 44I. Invalid files may be rejected; unknown relevance or unsupported disease appearance is uncertain, with no diagnosis.


In [ ]:
def predict_bark_disease(image_path, trained_model=best_model):
    try:
        _, batch = preprocess_single_image(image_path)
    except (OSError, ValueError, Image.DecompressionBombError):
        return {"status": "rejected", "prediction": None, "candidate_class": None,
                "confidence_score": None, "confidence_margin": None, "top_predictions": [],
                "reason": "invalid_or_unsupported_image", "review_recommended": True,
                "disease_classification_performed": False, "healthy_detection_supported": False,
                "is_cinnamon": None}
    return analyse_bark_batch(batch, trained_model=trained_model)


## CELL 61 — Test inference function

This function deliberately has no healthy result.

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
example_image = test_df.iloc[0]["path"]
result = predict_bark_disease(example_image)
print(json.dumps(result, indent=2))


## CELL 62 — Grad-CAM function

**Repair record:** Completed the truncated Grad-CAM graph and gradient calculation through the nested backbone and existing classifier head.


In [ ]:
def make_gradcam_heatmap(image_batch, trained_model, class_index=None):
    backbone_model = trained_model.get_layer(BACKBONE_NAME)
    grad_model = tf.keras.Model(backbone_model.input, [
        backbone_model.get_layer("top_conv").output, backbone_model.output,
    ])
    pooling_layer = trained_model.get_layer("global_average_pooling")
    dropout_layer = trained_model.get_layer("classification_dropout")
    prediction_layer = trained_model.get_layer("bark_predictions")
    image_tensor = tf.convert_to_tensor(image_batch)
    with tf.GradientTape() as tape:
        tape.watch(image_tensor)
        conv_outputs, features = grad_model(image_tensor, training=False)
        pooled = pooling_layer(features)
        probabilities = prediction_layer(dropout_layer(pooled, training=False))
        target_index = (int(tf.argmax(probabilities[0])) if class_index is None else int(class_index))
        class_score = probabilities[:, target_index]
    gradients = tape.gradient(class_score, conv_outputs)
    if gradients is None:
        raise RuntimeError("Grad-CAM gradients are unavailable for this graph.")
    pooled_gradients = tf.reduce_mean(gradients, axis=(0, 1, 2))
    heatmap = tf.reduce_sum(conv_outputs[0] * pooled_gradients, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = tf.math.divide_no_nan(heatmap, tf.reduce_max(heatmap))
    return heatmap.numpy(), probabilities[0].numpy()


## CELL 63 — Display a Grad-CAM example

Inspect model attention visually; Grad-CAM is not independent diagnostic validation.

**Repair record:** Completed the missing heatmap resize/overlay; retained the model softmax output and explained the visualization limit.


In [ ]:
example_image = test_df.iloc[0]["path"]
original_image, batch = preprocess_single_image(example_image)
heatmap, probabilities = make_gradcam_heatmap(batch, best_model)
prediction_index = int(np.argmax(probabilities))
print("Prediction:", CLASS_NAMES[prediction_index])
print("Probabilities:", probabilities)
heatmap_image = Image.fromarray(np.uint8(np.clip(heatmap, 0, 1) * 255))
heatmap_image = heatmap_image.resize(original_image.size, Image.Resampling.BILINEAR)
plt.figure(figsize=(7, 7))
plt.imshow(original_image)
plt.imshow(np.asarray(heatmap_image), cmap="jet", alpha=0.40, vmin=0, vmax=255)
plt.title(f"Grad-CAM: {CLASS_NAMES[prediction_index]}")
plt.axis("off")
plt.show()
print("A saliency map is an inspection aid, not independent proof of a diagnosis.")


## CELL 64 — Create final handoff directories

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
HANDOFF_DIR = Path("/content/bark_model_handoff")
if HANDOFF_DIR.exists():
    shutil.rmtree(HANDOFF_DIR)
EVALUATION_DIR = HANDOFF_DIR / "evaluation"
PARITY_DIR = HANDOFF_DIR / "parity_fixtures"
PARITY_IMAGES_DIR = PARITY_DIR / "images"
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
PARITY_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
print(HANDOFF_DIR)


## CELL 65 — Save final Keras model

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
KERAS_MODEL_PATH = HANDOFF_DIR / "bark_condition_model.keras"
best_model.save(KERAS_MODEL_PATH)
print("Saved:", KERAS_MODEL_PATH)


## CELL 66 — Save companion HDF5 model

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
H5_MODEL_PATH = HANDOFF_DIR / "bark_condition_model.h5"
best_model.save(H5_MODEL_PATH, include_optimizer=False)
print("Saved:", H5_MODEL_PATH)


## CELL 67 — Save exact class order

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
with open(HANDOFF_DIR / "class_names.json", "w", encoding="utf-8") as file:
    json.dump(CLASS_NAMES, file, indent=2)
print(CLASS_NAMES)


## CELL 68 — Save label definitions

**Repair record:** Completed the truncated label-definitions JSON writer; preserved the exact two labels.


In [ ]:
label_definitions = {
    "RoughBark": {"display_name": "Rough Bark", "type": "supported_disease_label",
                  "dataset_label": "RoughBark", "healthy": False},
    "StripeCanker": {"display_name": "Stripe Canker", "type": "supported_disease_label",
                     "dataset_label": "StripeCanker", "healthy": False},
}
with open(HANDOFF_DIR / "label_definitions.json", "w", encoding="utf-8") as file:
    json.dump(label_definitions, file, indent=2)


## CELL 69 — Save backend preprocessing.py

**Repair record:** Restored line breaks inside the generated Python string and added the missing file write.


In [ ]:
preprocessing_code = '''from PIL import Image, ImageOps
import numpy as np

IMAGE_SIZE = 224

def preprocess_bark_image(path_or_file):
    with Image.open(path_or_file) as image:
        image = ImageOps.exif_transpose(image).convert("RGB")
        image = image.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
        array = np.asarray(image, dtype=np.float32)
    return np.expand_dims(array, axis=0)
'''
(HANDOFF_DIR / "preprocessing.py").write_text(preprocessing_code, encoding="utf-8")


## CELL 70 — Save training environment

**Repair record:** Completed environment data and JSON writer; recorded actual package/platform/build versions instead of installing target-backend pins.


In [ ]:
training_environment = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(), "tensorflow": tf.__version__, "keras": keras.__version__,
    "numpy": np.__version__, "pandas": pd.__version__, "scikit_learn": sklearn.__version__,
    "pillow": PIL.__version__, "h5py": h5py.__version__,
    "kagglehub": package_metadata.version("kagglehub"),
    "matplotlib": package_metadata.version("matplotlib"),
    "platform": platform.platform(),
    "gpu_devices": [str(device) for device in tf.config.list_physical_devices("GPU")],
    "tensorflow_build": {key: str(value) for key, value in tf.sysconfig.get_build_info().items()},
}
with open(HANDOFF_DIR / "training_environment.json", "w", encoding="utf-8") as file:
    json.dump(training_environment, file, indent=2)
display(training_environment)


## CELL 71 — Save dataset summary

**Repair record:** Completed dataset-summary fields and writer; records excluded cross-label conflicts and exports their audit alongside near-duplicate/invalid-image audits.


In [ ]:
path_to_split = dict(zip(split_manifest["path"], split_manifest["split"]))
cross_split_near_pairs = int(sum(path_to_split.get(row.path_a) != path_to_split.get(row.path_b)
                                for row in near_duplicate_df.itertuples()))
dataset_summary = {
    "source": "Kaggle", "kaggle_dataset_id": KAGGLE_DATASET_ID,
    "dataset_directory": str(DATASET_DIR), "download_cache_path": str(DATASET_ROOT),
    "total_usable_images": int(len(df)), "invalid_images_removed": int(len(bad_images)),
    "exact_duplicates_removed": int(DUPLICATES_REMOVED),
    "conflicting_label_hash_groups_excluded": int(CONFLICTING_HASH_GROUPS_REMOVED),
    "conflicting_label_images_excluded": int(CONFLICTING_IMAGES_REMOVED),
    "conflict_resolution_policy": "Exclude every copy of a cross-label identical file; do not infer its correct label.",
    "potential_near_duplicate_pairs": int(len(near_duplicate_df)),
    "near_duplicate_audit_complete": bool(near_duplicate_audit_complete),
    "near_duplicate_cross_split_pairs_reported": cross_split_near_pairs,
    "class_counts": {name: int((df["class_name"] == name).sum()) for name in CLASS_NAMES},
    "partitions": {name: {"images": int(len(part)), "class_counts": {
        label: int((part["class_name"] == label).sum()) for label in CLASS_NAMES}}
        for name, part in [("train", train_df), ("validation", validation_df), ("test", test_df)]},
    "split_method": "stratified_image_level_after_exact_file_deduplication",
    "split_seed": SEED, "healthy_detection_supported": False,
    "tree_site_independence_established": False,
    "limitations": ["No healthy class is supplied.", "Tree/site IDs are not provided by this workflow.",
                    "File hashes do not detect all re-encoded duplicates.",
                    "Near-duplicate pairs are reported only; related-image leakage may remain.",
                    "Dataset disease labels have not been independently expert-confirmed by this notebook."],
}
with open(HANDOFF_DIR / "dataset_summary.json", "w", encoding="utf-8") as file:
    json.dump(dataset_summary, file, indent=2)
near_duplicate_df.to_csv(EVALUATION_DIR / "near_duplicate_audit.csv", index=False)
bad_images.to_csv(EVALUATION_DIR / "invalid_images.csv", index=False)
conflicting_images.to_csv(EVALUATION_DIR / "conflicting_label_images.csv", index=False)


## CELL 72 — Copy training history and split manifest

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
split_manifest.to_csv(HANDOFF_DIR / "split_manifest.csv", index=False)
training_history.to_csv(HANDOFF_DIR / "training_history.csv", index=False)


## CELL 73 — Save classification report

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
classification_report_dict = classification_report(
    y_test, test_predictions, labels=[0, 1], target_names=CLASS_NAMES, output_dict=True, zero_division=0,
)
with open(EVALUATION_DIR / "classification_report.json", "w", encoding="utf-8") as file:
    json.dump(classification_report_dict, file, indent=2)


## CELL 74 — Save test predictions

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
test_results.to_csv(EVALUATION_DIR / "test_predictions.csv", index=False)


## CELL 75 — Save confusion matrix CSV and PNG

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
confusion_df = pd.DataFrame(confusion, index=CLASS_NAMES, columns=CLASS_NAMES)
confusion_df.to_csv(EVALUATION_DIR / "confusion_matrix.csv")
fig, ax = plt.subplots(figsize=(7, 7))
display_object = ConfusionMatrixDisplay(confusion_matrix=confusion, display_labels=CLASS_NAMES)
display_object.plot(ax=ax, values_format="d")
plt.title("Cinnamon Bark Disease Confusion Matrix")
plt.savefig(EVALUATION_DIR / "confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.close(fig)


## CELL 76 — Save threshold calibration

**Repair record:** Completed the policy export and documented validation targets, fallback and unsupported healthy/OOD claims.

**Input-rejection update:** Export the final combined acceptance switch and the required reference-gate policy.

**Staged relevance update:** Require calibrated relevance acceptance alongside disease-score validation; the disease-distance gate is not an input classifier.


In [ ]:
threshold_policy = {
    "confidence_threshold": CONFIDENCE_THRESHOLD, "margin_threshold": MARGIN_THRESHOLD,
    "selected_using": "validation_data_only", "target_accepted_accuracy": TARGET_ACCEPTED_ACCURACY,
    "minimum_validation_coverage": MIN_VALIDATION_COVERAGE,
    "acceptance_enabled": bool(ACCEPTANCE_ENABLED), "validation_targets_met": bool(ACCEPTANCE_ENABLED),
    "selected_validation_coverage": float(selected_threshold["coverage"]),
    "selected_validation_accepted_accuracy": float(selected_threshold["accepted_accuracy"]),
    "accepted_result": "disease_detected", "unaccepted_result": "uncertain",
    "healthy_detection_supported": False,
    "final_acceptance_enabled": bool(ACCEPTANCE_ENABLED and RELEVANCE_POLICY["acceptance_enabled"]),
    "domain_policy_file": "ood_policy.json",
    "rejected_result": "rejected",
    "note": "These thresholds only accept RoughBark/StripeCanker classification. They do not distinguish healthy bark or reject all unknown inputs.",
    "fallback": "Disable acceptance when no validation policy meets the stated targets.",
}
with open(EVALUATION_DIR / "threshold_calibration.json", "w", encoding="utf-8") as file:
    json.dump(threshold_policy, file, indent=2)
threshold_df.to_csv(EVALUATION_DIR / "validation_threshold_search.csv", index=False)


## CELL 76A — Export rejection assets, calibration and test records

Export disease reference assets plus relevance model when trained, policy, splits, history and independent metrics. Never claim a relevance model exists when no such labelled dataset was supplied.


In [ ]:
OOD_REFERENCE_PATH = HANDOFF_DIR / "ood_reference.npz"
np.savez_compressed(OOD_REFERENCE_PATH, embeddings=train_embeddings_normalized,
                    class_ids=train_embedding_labels,
                    image_sha256=train_df["sha256"].to_numpy(dtype="U64"))
OOD_POLICY["reference_sha256"] = sha256_file(OOD_REFERENCE_PATH)
OOD_POLICY["model_sha256"] = {
    "bark_condition_model.keras": sha256_file(KERAS_MODEL_PATH),
    "bark_condition_model.h5": sha256_file(H5_MODEL_PATH),
}
OOD_POLICY["training_manifest_sha256"] = sha256_file(SPLIT_MANIFEST_PATH)
with open(HANDOFF_DIR / "ood_policy.json", "w", encoding="utf-8") as file:
    json.dump(OOD_POLICY, file, indent=2, allow_nan=False)
(HANDOFF_DIR / "bark_rejection.py").write_text(rejection_runtime_code, encoding="utf-8")
gate_calibration_df.to_csv(EVALUATION_DIR / "ood_validation_threshold_search.csv", index=False)
pd.DataFrame({"path": validation_df["path"].to_numpy(), "distance": validation_ood_distances,
              "reference_accepted": validation_in_domain}).to_csv(
                  EVALUATION_DIR / "ood_positive_validation.csv", index=False)
negative_validation_audit = negative_validation_df.copy()
negative_validation_audit["domain_distance"] = negative_validation_distances
negative_validation_audit.to_csv(EVALUATION_DIR / "ood_negative_validation.csv", index=False)
negative_test_results.to_csv(EVALUATION_DIR / "ood_negative_test.csv", index=False)
negative_manifest.to_csv(HANDOFF_DIR / "negative_split_manifest.csv", index=False)
pd.DataFrame(negative_excluded, columns=["path", "reason", "detail"]).to_csv(
    EVALUATION_DIR / "ood_invalid_negative_images.csv", index=False)
with open(EVALUATION_DIR / "ood_metrics.json", "w", encoding="utf-8") as file:
    json.dump(OOD_TEST_METRICS, file, indent=2, allow_nan=False)
print("Saved reference bank, shared runtime, policy, negative manifest and rejection evaluation.")

RELEVANCE_MODEL_PATH=None
relevance_export_parity_pass=None
if relevance_model is not None:
    RELEVANCE_MODEL_PATH=HANDOFF_DIR/"bark_relevance_model.keras"
    relevance_model.save(RELEVANCE_MODEL_PATH)
    relevance_reloaded=tf.keras.models.load_model(RELEVANCE_MODEL_PATH,compile=False)
    rsaved=relevance_reloaded.predict(make_dataset(relevance_parts["validation"],training=False),verbose=0).reshape(-1)
    relevance_export_parity_pass=bool(np.max(np.abs(rsaved-relevance_validation_scores))<=1e-5
        and all(assess_relevance_score(float(a),RELEVANCE_POLICY)["status"]==
                assess_relevance_score(float(b),RELEVANCE_POLICY)["status"]
                for a,b in zip(rsaved,relevance_validation_scores)))
    RELEVANCE_POLICY["model_file"]=RELEVANCE_MODEL_PATH.name
    RELEVANCE_POLICY["model_sha256"]=sha256_file(RELEVANCE_MODEL_PATH)
    RELEVANCE_POLICY["export_parity_passed"]=relevance_export_parity_pass
with open(HANDOFF_DIR/"relevance_policy.json","w",encoding="utf-8") as file:
    json.dump(RELEVANCE_POLICY,file,indent=2,allow_nan=False)
relevance_manifest.to_csv(HANDOFF_DIR/"relevance_split_manifest.csv",index=False)
relevance_history.to_csv(HANDOFF_DIR/"relevance_training_history.csv",index=False)
pd.DataFrame(relevance_threshold_rows).to_csv(EVALUATION_DIR/"relevance_validation_thresholds.csv",index=False)
relevance_test_predictions.to_csv(EVALUATION_DIR/"relevance_test_predictions.csv",index=False)
with open(EVALUATION_DIR/"relevance_metrics.json","w",encoding="utf-8") as file:
    json.dump({"training":relevance_training_summary,"test":relevance_test_metrics,
               "export_parity_passed":relevance_export_parity_pass},file,indent=2,allow_nan=False)


## CELL 77 — Save final metrics

**Repair record:** Completed truncated metrics with actual test support/coverage; unsupported health accuracy remains null.

**Input-rejection update:** Include held-out rejection metrics and distinguish absent negative testing from a measured result.

**v5 update:** The default `reference_only_experimental` mode uses the existing two disease folders. A missing relevance model does not block reference-matching experimental class predictions. Current executed code and the v5 overview supersede earlier strict-only notes.


In [ ]:
metrics = {
    "chosen_training_stage": chosen_stage,
    "stage1_validation_accuracy": float(stage1_accuracy), "stage1_validation_macro_f1": float(stage1_macro_f1),
    "stage2_validation_accuracy": float(stage2_accuracy), "stage2_validation_macro_f1": float(stage2_macro_f1),
    "test_accuracy": float(test_accuracy), "test_macro_f1": float(test_macro_f1),
    "test_balanced_accuracy": float(test_balanced_accuracy), "test_images": int(len(test_df)),
    "inference_mode": INFERENCE_MODE,
    "test_accepted_images": int(test_accepted.sum()),
    "test_uncertain_images": sum(r["status"] == "uncertain" for r in test_gate_results),
    "test_rejected_images": sum(r["status"] == "rejected" for r in test_gate_results),
    "test_experimental_candidates": int(test_experimental.sum()),
    "experimental_test_coverage": experimental_test_coverage,
    "experimental_test_accuracy": experimental_test_accuracy,
    "test_coverage": test_coverage, "accepted_test_accuracy": accepted_test_accuracy,
    "test_class_support": {name: int((y_test == i).sum()) for i, name in enumerate(CLASS_NAMES)},
    "ood_evaluation": OOD_TEST_METRICS,
    "healthy_detection_supported": False, "healthy_unhealthy_accuracy": None,
    "model_selection_source": "validation_macro_f1", "threshold_source": "validation_only",
}
with open(EVALUATION_DIR / "metrics.json", "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2, allow_nan=False)


## CELL 78 — Save error analysis

**Repair record:** Completed the truncated error-analysis writer and documented the retained split/domain limitations.

**Input-rejection update:** Record whether negative calibration/testing exists alongside the original disease errors.


In [ ]:
error_lines = ["# Bark Model Error Analysis", "", f"Selected model: {chosen_stage}", "",
               f"Test images: {len(test_df)}", f"Misclassified images: {len(mistakes)}", "",
               "## Misclassified images", ""]
if mistakes.empty:
    error_lines.append("No misclassified images were observed in this test partition.")
else:
    for _, row in mistakes.iterrows():
        error_lines.append(f"- {row['filename']}: true={row['class_name']}, predicted={row['predicted_class']}, "
                           f"confidence={row['confidence']:.4f}, margin={row['margin']:.4f}")
error_lines += ["", "## Scope and limitations", "",
                f"Cross-label identical images excluded before splitting: {CONFLICTING_IMAGES_REMOVED} ({CONFLICTING_HASH_GROUPS_REMOVED} hash groups).",
                f"Negative-validation targets met: {OOD_POLICY['negative_validation_targets_met']}.",
                f"Negative test images: {len(negative_test_results)}.",
                "Healthy bark is not supported; unknown uploads may still receive high scores.",
                "Image-level splitting does not establish unseen-tree/site generalization.",
                f"Potential near-duplicate cross-split pairs reported: {cross_split_near_pairs}.",
                "Do not change the model/thresholds after examining these test errors and reuse this test as independent."]
(EVALUATION_DIR / "error_analysis.md").write_text("\n".join(error_lines) + "\n", encoding="utf-8")


## CELL 79 — Create real-image parity fixture set

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.

**Input-rejection update:** Add held-out negative fixtures when supplied so rejection decisions are checked across serialization/runtime versions.


In [ ]:
fixture_parts = [test_df[test_df["class_name"] == class_name].head(4) for class_name in CLASS_NAMES]
if not negative_test_df.empty:
    negative_fixtures = negative_test_df.head(4).copy()
    negative_fixtures["class_name"] = "unsupported_negative"
    fixture_parts.append(negative_fixtures)
fixture_df = pd.concat(fixture_parts, ignore_index=True)
assert set(CLASS_NAMES).issubset(set(fixture_df["class_name"]))
display(fixture_df[["filename", "class_name"]])


## CELL 80 — Copy parity images

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
fixture_file_names = []
for index, row in fixture_df.iterrows():
    source_path = Path(row["path"])
    output_name = f"{index:02d}_{row['class_name']}_{source_path.name}"
    shutil.copy2(source_path, PARITY_IMAGES_DIR / output_name)
    fixture_file_names.append(output_name)
print("Parity images copied:", len(fixture_file_names))


## CELL 81 — Calculate original fixture predictions

**Repair record:** Completed fixture records with image/tensor hashes, repeated-inference checks and expected uncertainty decisions.

**Input-rejection update:** Record the expected distance and full rejected/uncertain/disease decision, not only softmax label agreement.

**Staged relevance update:** Parity fixtures/decisions now include the separate relevance result; raw disease scores remain serialization diagnostics only.


In [ ]:
fixture_expected = []
for index, row in fixture_df.iterrows():
    _, batch = preprocess_single_image(row["path"])
    outputs = [np.asarray(v) for v in analysis_model(batch, training=False)]
    probabilities, embeddings = outputs[0][0], outputs[1]
    repeated = np.asarray(analysis_model(batch, training=False)[0])[0]
    np.testing.assert_allclose(probabilities, repeated, atol=1e-5, rtol=0)
    distance = float(mean_knn_cosine_distance(embeddings, train_embeddings_normalized, k=OOD_K)[0])
    result = assess_prediction(probabilities,distance,OOD_POLICY,threshold_policy,relevance_for_batch(batch))
    fixture_expected.append({
        "file": fixture_file_names[index], "true_class": row["class_name"],
        "image_sha256": sha256_file(PARITY_IMAGES_DIR / fixture_file_names[index]),
        "input_tensor_sha256": hashlib.sha256(batch.astype("<f4").tobytes(order="C")).hexdigest(),
        "expected_probabilities": probabilities.tolist(),
        "expected_relevance_score": relevance_for_batch(batch)["score"],
        "raw_classifier_class": CLASS_NAMES[int(np.argmax(probabilities))],
        "predicted_class": result["candidate_class"],
        "expected_disease_classification_performed": result["disease_classification_performed"],
        "expected_domain_distance": distance, "expected_status": result["status"],
        "expected_prediction": result["prediction"], "expected_reason": result["reason"],
        "expected_accepted": result["status"] == "disease_detected",
        "expected_assessment_mode": result["assessment_mode"],
        "expected_input_relevance_verified": result["input_relevance_verified"],
    })


## CELL 82 — Reload .keras and compare predictions

**Repair record:** Completed the truncated difference loop and missing aggregate variables; also checks uncertainty-policy agreement.

**Input-rejection update:** Compare exported classifier features, reference distances and full decisions with the same training reference bank.

**Staged relevance update:** Parity fixtures/decisions now include the separate relevance result; raw disease scores remain serialization diagnostics only.


In [ ]:
keras_reloaded = tf.keras.models.load_model(KERAS_MODEL_PATH, compile=False)
keras_analysis = build_bark_analysis_model(keras_reloaded)
keras_differences, keras_label_matches, keras_policy_matches = [], [], []
keras_distance_differences = []
for fixture, (_, row) in zip(fixture_expected, fixture_df.iterrows()):
    _, batch = preprocess_single_image(row["path"])
    original_prediction = np.asarray(fixture["expected_probabilities"], dtype=np.float32)
    probabilities, embeddings = [np.asarray(v) for v in keras_analysis(batch, training=False)]
    reloaded_prediction = probabilities[0]
    distance = float(mean_knn_cosine_distance(embeddings, train_embeddings_normalized, k=OOD_K)[0])
    result = assess_prediction(reloaded_prediction,distance,OOD_POLICY,threshold_policy,relevance_for_batch(batch))
    keras_differences.append(float(np.max(np.abs(original_prediction - reloaded_prediction))))
    keras_distance_differences.append(abs(distance - fixture["expected_domain_distance"]))
    keras_label_matches.append(result["candidate_class"] == fixture["predicted_class"]
        and CLASS_NAMES[int(np.argmax(reloaded_prediction))] == fixture["raw_classifier_class"])
    keras_policy_matches.append(result["status"] == fixture["expected_status"]
        and result["prediction"] == fixture["expected_prediction"] and result["reason"] == fixture["expected_reason"]
        and result["disease_classification_performed"] == fixture["expected_disease_classification_performed"])
KERAS_MAX_DIFFERENCE = float(max(keras_differences))
KERAS_MAX_DISTANCE_DIFFERENCE = float(max(keras_distance_differences))
KERAS_LABEL_AGREEMENT = float(np.mean(keras_label_matches))
KERAS_POLICY_AGREEMENT = float(np.mean(keras_policy_matches))
print(".keras score/distance maximum differences:", KERAS_MAX_DIFFERENCE, KERAS_MAX_DISTANCE_DIFFERENCE)


## CELL 83 — Reload .h5 and compare predictions

**Repair record:** Completed the truncated HDF5 parity loop and missing aggregate variables; also checks uncertainty-policy agreement.

**Input-rejection update:** Compare exported classifier features, reference distances and full decisions with the same training reference bank.

**Staged relevance update:** Parity fixtures/decisions now include the separate relevance result; raw disease scores remain serialization diagnostics only.


In [ ]:
h5_reloaded = tf.keras.models.load_model(H5_MODEL_PATH, compile=False)
h5_analysis = build_bark_analysis_model(h5_reloaded)
h5_differences, h5_label_matches, h5_policy_matches = [], [], []
h5_distance_differences = []
for fixture, (_, row) in zip(fixture_expected, fixture_df.iterrows()):
    _, batch = preprocess_single_image(row["path"])
    original_prediction = np.asarray(fixture["expected_probabilities"], dtype=np.float32)
    probabilities, embeddings = [np.asarray(v) for v in h5_analysis(batch, training=False)]
    reloaded_prediction = probabilities[0]
    distance = float(mean_knn_cosine_distance(embeddings, train_embeddings_normalized, k=OOD_K)[0])
    result = assess_prediction(reloaded_prediction,distance,OOD_POLICY,threshold_policy,relevance_for_batch(batch))
    h5_differences.append(float(np.max(np.abs(original_prediction - reloaded_prediction))))
    h5_distance_differences.append(abs(distance - fixture["expected_domain_distance"]))
    h5_label_matches.append(result["candidate_class"] == fixture["predicted_class"]
        and CLASS_NAMES[int(np.argmax(reloaded_prediction))] == fixture["raw_classifier_class"])
    h5_policy_matches.append(result["status"] == fixture["expected_status"]
        and result["prediction"] == fixture["expected_prediction"] and result["reason"] == fixture["expected_reason"]
        and result["disease_classification_performed"] == fixture["expected_disease_classification_performed"])
H5_MAX_DIFFERENCE = float(max(h5_differences))
H5_MAX_DISTANCE_DIFFERENCE = float(max(h5_distance_differences))
H5_LABEL_AGREEMENT = float(np.mean(h5_label_matches))
H5_POLICY_AGREEMENT = float(np.mean(h5_policy_matches))
print(".h5 score/distance maximum differences:", H5_MAX_DIFFERENCE, H5_MAX_DISTANCE_DIFFERENCE)


## CELL 84 — Evaluate export parity

Do not increase tolerance merely to force a pass. Failed formats are retained for diagnosis, not deployment.

**Repair record:** Completed the truncated parity status display and policy checks; reports failures honestly while allowing the diagnostic ZIP to be created.

**Input-rejection update:** Require numerical reference-distance parity in addition to probabilities, labels and full decision agreement.

**Staged relevance update:** Report relevance export parity alongside disease-model parity.


In [ ]:
PARITY_TOLERANCE = 1e-5
keras_parity_pass = bool(KERAS_MAX_DIFFERENCE <= PARITY_TOLERANCE
                         and KERAS_MAX_DISTANCE_DIFFERENCE <= PARITY_TOLERANCE
                         and KERAS_LABEL_AGREEMENT == 1.0 and KERAS_POLICY_AGREEMENT == 1.0)
h5_parity_pass = bool(H5_MAX_DIFFERENCE <= PARITY_TOLERANCE
                      and H5_MAX_DISTANCE_DIFFERENCE <= PARITY_TOLERANCE
                      and H5_LABEL_AGREEMENT == 1.0 and H5_POLICY_AGREEMENT == 1.0)
print(".keras parity:", "PASS" if keras_parity_pass else "REVIEW")
print(".h5 parity:", "PASS" if h5_parity_pass else "REVIEW")
print("Failed artifacts remain in the diagnostic package, but are not approved for deployment.")
print("Do not increase the tolerance just to make an export pass.")
if relevance_export_parity_pass is False:
    print("REVIEW: relevance export changed predictions; do not deploy this package.")


## CELL 85 — Save parity information

**Repair record:** Completed the parity JSON writer; records actual format results and explicitly leaves backend parity untested.

**Input-rejection update:** Include the distance-parity measurements and reference-policy filename in the handoff fixtures.


In [ ]:
parity_information = {
    "same_environment_tolerance": PARITY_TOLERANCE,
    "ood_policy_file": "ood_policy.json",
    "class_names": CLASS_NAMES, "decision_policy": threshold_policy,
    "keras": {"maximum_absolute_difference": KERAS_MAX_DIFFERENCE,
              "maximum_distance_difference": KERAS_MAX_DISTANCE_DIFFERENCE,
              "label_agreement": KERAS_LABEL_AGREEMENT, "policy_agreement": KERAS_POLICY_AGREEMENT,
              "passed": keras_parity_pass},
    "h5": {"maximum_absolute_difference": H5_MAX_DIFFERENCE,
           "maximum_distance_difference": H5_MAX_DISTANCE_DIFFERENCE,
           "label_agreement": H5_LABEL_AGREEMENT, "policy_agreement": H5_POLICY_AGREEMENT,
           "passed": h5_parity_pass},
    "backend_parity_tested": False, "fixtures": fixture_expected,
}
with open(PARITY_DIR / "expected_predictions.json", "w", encoding="utf-8") as file:
    json.dump(parity_information, file, indent=2, allow_nan=False)


## CELL 86 — Save target CinnaAI backend requirements

These are the previously inspected backend target versions. This cell does not install them in Colab.

**Repair record:** Restored actual newlines between requirements. Target-backend versions are documentation only, not a Colab installation command.


In [ ]:
target_backend_requirements = "\n".join([
    "tensorflow==2.16.2", "keras==3.15.1", "numpy==1.26.4", "h5py==3.16.0", "Pillow==12.2.0",
]) + "\n"
(HANDOFF_DIR / "requirements-inference.txt").write_text(target_backend_requirements, encoding="utf-8")
print("Documented target backend versions; no packages were installed or downgraded by this cell.")


## CELL 87 — Save actual Colab training requirements

**Repair record:** Restored requirement-file newlines and records actual KaggleHub/Matplotlib versions used by training.


In [ ]:
training_requirements = "\n".join([
    f"tensorflow=={tf.__version__}", f"keras=={keras.__version__}", f"numpy=={np.__version__}",
    f"pandas=={pd.__version__}", f"scikit-learn=={sklearn.__version__}", f"Pillow=={PIL.__version__}",
    f"h5py=={h5py.__version__}", f"kagglehub=={training_environment['kagglehub']}",
    f"matplotlib=={training_environment['matplotlib']}",
]) + "\n"
(HANDOFF_DIR / "requirements-training.txt").write_text(training_requirements, encoding="utf-8")


## CELL 88 — Calculate important SHA-256 checksums

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
def sha256_path(file_path):
    hasher = hashlib.sha256()
    with open(file_path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            hasher.update(block)
    return hasher.hexdigest()

KERAS_SHA256 = sha256_path(KERAS_MODEL_PATH)
H5_SHA256 = sha256_path(H5_MODEL_PATH)
SPLIT_SHA256 = sha256_path(HANDOFF_DIR / "split_manifest.csv")


## CELL 89 — Create metadata.json

**Repair record:** Completed the truncated metadata with preprocessing/output/versions/policy/hashes. No backend or deployment success is asserted before testing.

**Input-rejection update:** Version the gated system and declare its required inference assets; deployment and species/healthy claims remain unsupported.

**Domain-first/display update:** Version the domain-first response contract so a Drive backup can detect an outdated pre-fix handoff.

**Staged relevance update:** Record staged relevance and disease-support semantics; missing relevance training is not misrepresented as a functioning species detector.

**v5 update:** The default `reference_only_experimental` mode uses the existing two disease folders. A missing relevance model does not block reference-matching experimental class predictions. Current executed code and the v5 overview supersede earlier strict-only notes.


In [ ]:
metadata = {
    "schema_version": 1, "model_version": "bark-disease-v5-two-class-experimental",
    "task": "supported_bark_disease_classification", "architecture": "EfficientNetB0",
    "model_file": "bark_condition_model.keras", "artifact_sha256": KERAS_SHA256,
    "companion_h5_sha256": H5_SHA256,
    "dataset": {"source": "Kaggle", "dataset_id": KAGGLE_DATASET_ID},
    "input": {"shape": [None, 224, 224, 3], "dtype": "float32", "colour_order": "RGB",
              "pixel_range": [0, 255], "exif_transpose": True, "resize_library": "Pillow",
              "resize_method": "bilinear", "aspect_ratio_policy": "stretch",
              "external_normalization": "none", "backbone_preprocessing": "included_in_model"},
    "output": {"shape": [None, 2], "activation": "softmax", "class_names": CLASS_NAMES},
    "decision_policy": threshold_policy,
    "input_rejection": OOD_POLICY,
    "relevance_gate": RELEVANCE_POLICY,
    "relevance_test_metrics": relevance_test_metrics,
    "prediction_runtime": {"domain_first": True, "skip_disease_head_for_rejected": True,
                           "hide_disease_scores_for_rejected": True,
                           "response_schema": "bark-prediction-v5",
                           "inference_mode": INFERENCE_MODE,
                           "separate_relevance_gate": relevance_model is not None,
                           "experimental_fallback_without_relevance": INFERENCE_MODE == "reference_only_experimental", "outside_disease_reference_status": "uncertain"},
    "required_rejection_files": ["bark_rejection.py", "ood_reference.npz", "ood_policy.json", "relevance_policy.json"]
        + (["bark_relevance_model.keras"] if relevance_model is not None else []),
    "training": {"seed": SEED, "chosen_stage": chosen_stage, "split_unit": "image",
                 "split_manifest_sha256": SPLIT_SHA256, "fine_tune_fraction": FINE_TUNE_FRACTION,
                 "checkpoint_selection": "validation_loss", "stage_selection": "validation_macro_f1"},
    "training_versions": training_environment,
    "target_backend_versions": {"tensorflow": "2.16.2", "keras": "3.15.1", "numpy": "1.26.4",
                                "h5py": "3.16.0", "pillow": "12.2.0"},
    "tested_backend_versions": None, "backend_parity_tested": False,
    "export_parity": {"keras_passed": keras_parity_pass, "h5_passed": h5_parity_pass},
    "deployment_ready": False,
    "deployment_note": "Backend parity and target-domain validation remain required; failed exports are diagnostic only.",
    "healthy_detection_supported": False, "evaluation": metrics,
    "limitations": dataset_summary["limitations"] + [
        "Confidence does not establish healthy bark or reliable rejection of unrelated images.",
        "Reference-only experimental candidates do not verify input relevance or constitute accepted diagnoses.",
        "No production backend inference has been tested by this Colab run.",
    ],
}
with open(HANDOFF_DIR / "metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2, allow_nan=False)


## CELL 90 — Create backend parity test script

**Repair record:** Completed the truncated generated parity script and writer. Supports either export and verifies model/image/tensor hashes, scores, labels and policy decisions.

**Input-rejection update:** Verify reference/model binding and complete rejection decisions in the target backend, using the same exported NumPy runtime.

**Staged relevance update:** Backend parity checks both relevance scores and final staged decisions, including unknown relevance and head-skipping on uncertain disease-reference outliers.

**v5 update:** The default `reference_only_experimental` mode uses the existing two disease folders. A missing relevance model does not block reference-matching experimental class predictions. Current executed code and the v5 overview supersede earlier strict-only notes.


In [ ]:
backend_parity_script = r'''import argparse
import hashlib
import json
from pathlib import Path
import numpy as np
import tensorflow as tf
from preprocessing import preprocess_bark_image
from bark_rejection import load_rejection_assets, mean_knn_cosine_distance, assess_prediction, predict_with_domain_gate, assess_relevance_score

BASE_DIR = Path(__file__).resolve().parent
parser = argparse.ArgumentParser(description="Check bark model probabilities AND rejection decisions.")
parser.add_argument("--model", default="bark_condition_model.keras")
args = parser.parse_args()
MODEL_PATH = BASE_DIR / args.model
with open(BASE_DIR / "parity_fixtures" / "expected_predictions.json", encoding="utf-8") as file:
    parity_data = json.load(file)
with open(BASE_DIR / "class_names.json", encoding="utf-8") as file:
    class_names = json.load(file)
assert class_names == ["RoughBark", "StripeCanker"] == parity_data["class_names"]
reference, domain_policy = load_rejection_assets(BASE_DIR, MODEL_PATH)
with open(BASE_DIR / "relevance_policy.json",encoding="utf-8") as file:
    relevance_policy=json.load(file)
relevance_model=None
if relevance_policy["available"]:
    relevance_path=BASE_DIR/relevance_policy["model_file"]
    assert hashlib.sha256(relevance_path.read_bytes()).hexdigest()==relevance_policy["model_sha256"]
    relevance_model=tf.keras.models.load_model(relevance_path,compile=False)
relevance_predictor=None if relevance_model is None else lambda images: np.asarray(relevance_model(images,training=False))
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
assert model.input_shape == (None, 224, 224, 3) and model.output_shape == (None, 2)
analysis = tf.keras.Model(model.inputs, [model.output, model.get_layer("global_average_pooling").output])
TOLERANCE = 1e-4
differences, distance_differences = [], []
for fixture in parity_data["fixtures"]:
    image_path = BASE_DIR / "parity_fixtures" / "images" / fixture["file"]
    assert hashlib.sha256(image_path.read_bytes()).hexdigest() == fixture["image_sha256"]
    batch = preprocess_bark_image(image_path)
    assert hashlib.sha256(batch.astype("<f4").tobytes(order="C")).hexdigest() == fixture["input_tensor_sha256"]
    probabilities, embeddings = [np.asarray(v) for v in analysis(batch, training=False)]
    actual = probabilities[0]
    expected = np.asarray(fixture["expected_probabilities"], dtype=np.float32)
    np.testing.assert_allclose(actual, expected, atol=TOLERANCE, rtol=0)
    np.testing.assert_allclose(actual, np.asarray(analysis(batch, training=False)[0])[0], atol=TOLERANCE, rtol=0)
    distance = float(mean_knn_cosine_distance(embeddings, reference, k=domain_policy["k"])[0])
    np.testing.assert_allclose(distance, fixture["expected_domain_distance"], atol=TOLERANCE, rtol=0)
    relevance_score=None if relevance_model is None else float(np.asarray(relevance_model(batch,training=False)).reshape(-1)[0])
    if fixture["expected_relevance_score"] is None:
        assert relevance_score is None
    else:
        np.testing.assert_allclose(relevance_score,fixture["expected_relevance_score"],atol=TOLERANCE,rtol=0)
    result=assess_prediction(actual,distance,domain_policy,parity_data["decision_policy"],
        assess_relevance_score(relevance_score,relevance_policy))
    # Exercise the actual head-skipping public path as well as raw diagnostic serialization.
    head_calls = []
    def run_head(features):
        head_calls.append(True)
        values = model.get_layer("classification_dropout")(tf.convert_to_tensor(features), training=False)
        return np.asarray(model.get_layer("bark_predictions")(values))
    public_result = predict_with_domain_gate(batch, lambda images: embeddings, run_head,
        reference,domain_policy,parity_data["decision_policy"],relevance_predictor,relevance_policy)
    assert bool(head_calls) == public_result["disease_classification_performed"]
    assert public_result["status"] == result["status"]
    assert public_result["prediction"] == result["prediction"]
    assert result["disease_classification_performed"] == fixture["expected_disease_classification_performed"]
    assert class_names[int(np.argmax(actual))] == fixture["raw_classifier_class"]
    assert result["candidate_class"] == fixture["predicted_class"]
    assert result["status"] == fixture["expected_status"], "Rejected/uncertain/experimental/disease status changed."
    assert result["prediction"] == fixture["expected_prediction"]
    assert result["assessment_mode"] == fixture["expected_assessment_mode"]
    assert result["input_relevance_verified"] == fixture["expected_input_relevance_verified"]
    assert result["reason"] == fixture["expected_reason"]
    differences.append(float(np.max(np.abs(actual - expected))))
    distance_differences.append(abs(distance - fixture["expected_domain_distance"]))
print("PASS:", MODEL_PATH.name, "fixtures:", len(differences),
      "score difference:", max(differences), "distance difference:", max(distance_differences))
print("Numerical compatibility only; field rejection/diagnostic reliability needs independent data.")
'''
(HANDOFF_DIR / "backend_parity_check.py").write_text(backend_parity_script, encoding="utf-8")


## CELL 91 — Create README for Codex

**Repair record:** Completed the truncated handoff README, including results, exact input/output, failed-export handling and integration instructions.

**Input-rejection update:** Explain which additional assets are needed to avoid using softmax alone and disclose missing/failed negative validation.

**Domain-first/display update:** Document domain-first inference and hidden rejected-image disease scores in the downloadable handoff.

**Staged relevance update:** Document separate gate assets, unknown relevance and disease-reference uncertainty in the saved package.

**v5 update:** The default `reference_only_experimental` mode uses the existing two disease folders. A missing relevance model does not block reference-matching experimental class predictions. Current executed code and the v5 overview supersede earlier strict-only notes.


In [ ]:
readme_text = f"""# CinnaAI two-class bark assessment v5

## Disease classifier
Exact class order: {CLASS_NAMES}. Selected stage: {chosen_stage}.
Raw held-out accuracy: {test_accuracy:.4f}; macro-F1: {test_macro_f1:.4f}.
Relevance-verified accepted coverage: {test_coverage:.4f}; accepted accuracy: {accepted_test_accuracy}.
Experimental candidate coverage: {experimental_test_coverage:.4f}; accuracy: {experimental_test_accuracy}.
The model has no healthy disease class.

## Decision order
Active mode: {INFERENCE_MODE}.
1. Invalid decoding or optional calibrated relevance rejection: rejected.
2. Outside the two-disease reference or inadequate disease evidence: uncertain.
3. In reference_only_experimental mode, missing relevance + reference pass + disease-score pass:
   experimental_candidate, with prediction/class scores and input_relevance_verified=false.
4. With a calibrated relevance pass + reference pass + disease-score pass: disease_detected.
Strict staged_relevance mode keeps missing/unconfirmed relevance uncertain.
Experimental candidates always recommend review; they are not accepted diagnoses.

A valid bark image is not automatically RoughBark or StripeCanker. A disease-reference mismatch
must not be presented as proof that a bark image is invalid. No arbitrary distance multiplier is used.
Input relevance model available: {RELEVANCE_POLICY['available']}.
Relevance acceptance calibrated: {RELEVANCE_POLICY['acceptance_enabled']}.
Relevance rejection calibrated: {RELEVANCE_POLICY['rejection_enabled']}.
The original two disease folders are enough for the default experimental workflow.
A separate relevance dataset/model is optional; its absence does not block experimental class scores.
Unrelated-image rejection accuracy is unmeasured when there is no held-out negative dataset.

## Runtime assets
Both disease formats are included. Rejection/support logic additionally needs bark_rejection.py,
ood_reference.npz, ood_policy.json and relevance_policy.json. If the relevance policy says available,
bark_relevance_model.keras is required. A classifier file by itself still forces one of two labels.
All models use EXIF-corrected RGB, 224x224 Pillow bilinear, float32 values 0–255;
application-specific preprocessing is inside each backbone. No external /255 is used.

## Validation and export
Disease .keras parity passed: {keras_parity_pass}; .h5 parity passed: {h5_parity_pass}.
Relevance export parity: {relevance_export_parity_pass}.
Independent relevance test: {relevance_test_metrics}.
Failed exports remain diagnostic and are not ready for deployment. Backend parity remains untested
until backend_parity_check.py runs in the real target backend.
python backend_parity_check.py
python backend_parity_check.py --model bark_condition_model.h5

## Scope and provenance
Dataset: {KAGGLE_DATASET_ID}.
Cross-label disease images excluded: {CONFLICTING_IMAGES_REMOVED}.
Disease split is image-level; tree/site independence is not established.
Relevance splits are supplied by the user; exact hashes are checked, but group independence is unverified.
Healthy-looking/unlabelled cinnamon bark may have a relevance label, never an invented disease label.
checksums.sha256 covers package files. Keep all assets together when using Drive backups.
"""
(HANDOFF_DIR/"README.md").write_text(readme_text,encoding="utf-8")


## CELL 92 — Create complete file checksums

All output files must exist before checksums. Do not modify files after this cell without rerunning checksums and ZIP creation.

**Repair record:** Completed the truncated checksum writer, with real newlines and portable relative paths.


In [ ]:
checksum_lines = []
for file_path in sorted(HANDOFF_DIR.rglob("*")):
    if file_path.is_file() and file_path.name != "checksums.sha256":
        relative_path = file_path.relative_to(HANDOFF_DIR).as_posix()
        checksum_lines.append(f"{sha256_path(file_path)}  {relative_path}")
(HANDOFF_DIR / "checksums.sha256").write_text("\n".join(checksum_lines) + "\n", encoding="utf-8")
print("Checksummed files:", len(checksum_lines))


## CELL 93 — Display final handoff package

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
print("FINAL MODEL HANDOFF CONTENTS")
print("=" * 70)
for file_path in sorted(HANDOFF_DIR.rglob("*")):
    if file_path.is_file():
        print(file_path.relative_to(HANDOFF_DIR))


## CELL 94 — Create final ZIP

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
FINAL_ZIP_PATH = shutil.make_archive("/content/bark_model_handoff", "zip", root_dir=HANDOFF_DIR)
print("Final ZIP created:", FINAL_ZIP_PATH)
print("Save the ZIP before disconnecting or resetting this Colab runtime.")


## CELL 94A — Save the verified model package to Google Drive

Set `SAVE_TO_GOOGLE_DRIVE=True` and run this cell after the updated CELLS 64–94.
It saves both model formats, preprocessing, rejection runtime/reference/policy, evaluation,
environment records and parity fixtures to a new UTC timestamp folder under
`MyDrive/CinnaAI/bark_models/`, plus a freshly built ZIP. No previous backup is overwritten.
Source checksums, copied-file checksums and the ZIP contents are verified before reporting success.
An outdated pre-fix handoff is rejected; rerun the updated export cells first.
This is optional and disabled by default so Run all does not pause for Drive authentication.
The manual uploaded photo is not added to the model package.

**Staged relevance update:** Include the optional relevance model and mandatory relevance policy/split/evaluation in the complete verified Drive backup.

**v5 update:** The default `reference_only_experimental` mode uses the existing two disease folders. A missing relevance model does not block reference-matching experimental class predictions. Current executed code and the v5 overview supersede earlier strict-only notes.


In [ ]:
backup_helper_code = r'''"""Verify and back up a complete generated handoff without overwriting older runs."""
import hashlib
import json
from pathlib import Path
from datetime import datetime, timezone
import shutil
import zipfile

REQUIRED_BACKUP_FILES = {
    "bark_condition_model.keras", "bark_condition_model.h5", "bark_rejection.py",
    "ood_reference.npz", "ood_policy.json", "relevance_policy.json", "relevance_split_manifest.csv",
    "evaluation/relevance_metrics.json", "metadata.json", "preprocessing.py",
    "class_names.json", "label_definitions.json", "backend_parity_check.py",
    "training_environment.json", "requirements-inference.txt", "requirements-training.txt",
    "README.md", "dataset_summary.json", "training_history.csv", "split_manifest.csv",
    "negative_split_manifest.csv", "evaluation/classification_report.json",
    "evaluation/test_predictions.csv", "evaluation/confusion_matrix.csv",
    "evaluation/confusion_matrix.png", "evaluation/error_analysis.md",
    "evaluation/metrics.json", "evaluation/ood_metrics.json",
    "evaluation/threshold_calibration.json", "parity_fixtures/expected_predictions.json",
}

def backup_file_hash(path):
    hasher = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

def verify_bark_handoff(directory):
    directory = Path(directory)
    checksum_file = directory / "checksums.sha256"
    if not checksum_file.is_file():
        raise FileNotFoundError("Run the updated export/checksum cells through CELL 94 first.")
    expected = {}
    for line in checksum_file.read_text(encoding="utf-8").splitlines():
        digest, relative = line.split("  ", 1)
        relative_path = Path(relative)
        if (len(digest) != 64 or relative_path.is_absolute() or ".." in relative_path.parts
                or relative == "checksums.sha256" or relative in expected):
            raise ValueError("Invalid handoff checksum index.")
        int(digest, 16)
        expected[relative] = digest
    actual = {file.relative_to(directory).as_posix() for file in directory.rglob("*")
              if file.is_file() and file != checksum_file}
    if actual != set(expected) or not REQUIRED_BACKUP_FILES.issubset(actual):
        raise ValueError("Handoff has missing/unindexed files. Rerun updated export/checksum cells before backup.")
    for relative, digest in expected.items():
        if backup_file_hash(directory / relative) != digest:
            raise ValueError(f"Handoff checksum mismatch: {relative}")
    with open(directory / "metadata.json", encoding="utf-8") as file:
        metadata = json.load(file)
    if not metadata.get("prediction_runtime", {}).get("domain_first", False):
        raise ValueError("This is a pre-fix handoff. Run the updated export cells before saving to Drive.")
    with open(directory / "ood_policy.json", encoding="utf-8") as file:
        policy = json.load(file)
    for name in ["bark_condition_model.keras", "bark_condition_model.h5"]:
        if policy["model_sha256"].get(name) != expected[name]:
            raise ValueError("Rejection policy and model hashes differ.")
    if policy["reference_sha256"] != expected["ood_reference.npz"]:
        raise ValueError("Rejection policy and reference hashes differ.")
    with open(directory / "relevance_policy.json",encoding="utf-8") as file:
        relevance=json.load(file)
    runtime = metadata.get("prediction_runtime",{})
    if runtime.get("response_schema") != "bark-prediction-v5":
        raise ValueError("Pre-v5 handoff; regenerate current exports before backup.")
    mode = policy.get("inference_mode")
    if (mode not in {"reference_only_experimental","staged_relevance"}
            or runtime.get("inference_mode") != mode):
        raise ValueError("Backup metadata and inference policy modes differ.")
    if relevance["available"]:
        model_file=relevance["model_file"]
        if expected.get(model_file)!=relevance["model_sha256"]:
            raise ValueError("Relevance model/policy hash mismatch.")
    return expected

def save_bark_handoff_backup(source, drive_root):
    source, drive_root = Path(source), Path(drive_root)
    expected = verify_bark_handoff(source)
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f_UTC")
    destination = drive_root / timestamp
    staging = drive_root / (timestamp + ".incomplete")
    staging.mkdir(parents=True, exist_ok=False)
    copied = staging / "bark_model_handoff"
    shutil.copytree(source, copied)
    verify_bark_handoff(copied)
    # Regenerate from the verified copy, rather than trusting a possibly stale ZIP.
    archive = Path(shutil.make_archive(str(staging / "bark_model_handoff"), "zip", root_dir=copied))
    with zipfile.ZipFile(archive) as zipped:
        archived_files = {name for name in zipped.namelist() if not name.endswith("/")}
        if archived_files != set(expected) | {"checksums.sha256"} or zipped.testzip() is not None:
            raise ValueError("Backup ZIP integrity/coverage check failed.")
        for relative, digest in expected.items():
            hasher = hashlib.sha256()
            with zipped.open(relative) as file:
                for chunk in iter(lambda: file.read(1024 * 1024), b""):
                    hasher.update(chunk)
            if hasher.hexdigest() != digest:
                raise ValueError(f"Backup ZIP checksum mismatch: {relative}")
    staging.rename(destination)
    return destination, destination / "bark_model_handoff.zip"
'''
exec(compile(backup_helper_code, "bark_drive_backup.py", "exec"), globals())

# Enable after the updated export/checksum/ZIP cells have completed.
SAVE_TO_GOOGLE_DRIVE = False
if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    # Validate locally before requesting a Drive mount.
    verify_bark_handoff(Path("/content/bark_model_handoff"))
    drive.mount("/content/drive")
    DRIVE_BACKUP_DIR, DRIVE_BACKUP_ZIP = save_bark_handoff_backup(
        Path("/content/bark_model_handoff"),
        Path("/content/drive/MyDrive/CinnaAI/bark_models"),
    )
    print("Verified model + rejection package saved to:", DRIVE_BACKUP_DIR)
    print("Verified ZIP:", DRIVE_BACKUP_ZIP)
else:
    print("Drive backup skipped. Set SAVE_TO_GOOGLE_DRIVE=True in CELL 94A and run this cell to save the complete verified package.")


## CELL 95 — Download final model package

Download and preserve `bark_model_handoff.zip`.

**Repair record:** Restored lost line breaks and Python indentation; retained the supplied operations.


In [ ]:
from google.colab import files
files.download(FINAL_ZIP_PATH)


## CELL 96 — Optional manual prediction test

Optional: enable the flag after training and run CELL 96 followed by CELL 97. This is not a new independent evaluation.

**Repair record:** Guarded the optional interactive upload so top-to-bottom Run all does not block on an unnecessary manual test.


In [ ]:
from google.colab import files
# Disabled by default so Run all completes without waiting for an optional upload.
RUN_OPTIONAL_MANUAL_TEST = False
uploaded = files.upload() if RUN_OPTIONAL_MANUAL_TEST else {}
if not RUN_OPTIONAL_MANUAL_TEST:
    print("Optional test skipped. Set RUN_OPTIONAL_MANUAL_TEST=True and run CELL 96 then CELL 97 to upload an image.")


## CELL 97 — Predict uploaded image

Display rejected, uncertain and disease_detected separately. Known-disease mismatch is uncertainty, not invalidity; scores appear only if the disease head was actually used.

**v5 update:** The default `reference_only_experimental` mode uses the existing two disease folders. A missing relevance model does not block reference-matching experimental class predictions. Current executed code and the v5 overview supersede earlier strict-only notes.


In [ ]:
def display_bark_prediction(result):
    print("\n" + "=" * 70)
    print("MODEL RESULT")
    print("=" * 70)
    print(json.dumps(result, indent=2))
    print("\n" + "=" * 70)
    print("FINAL INTERPRETATION")
    print("=" * 70)
    status = result.get("status")
    if status == "rejected":
        print("IMAGE REJECTED")
        if result.get("reason") == "invalid_or_unsupported_image":
            print("This file could not be decoded as a supported image.")
        else:
            print("The calibrated relevance model marked this image unsuitable for bark assessment.")
        if result.get("domain_distance") is not None:
            print(f"Domain distance: {result['domain_distance']:.4f}")
            print(f"Maximum accepted distance: {result['domain_distance_threshold']:.4f}")
        print("No disease classification was performed.")
    elif status == "uncertain":
        print("UNCERTAIN — no disease diagnosis accepted")
        if result.get("reason") == "outside_known_disease_bark_reference":
            print("The image does not match the supported disease reference. This does not establish that it is invalid or healthy.")
        elif result.get("reason") == "relevance_model_unavailable":
            print("A separate relevance model has not been trained. The two disease classes cannot establish input suitability.")
        elif result.get("reason") == "relevance_unconfirmed_or_calibration_unmet":
            print("Image relevance is not confident or its validation targets were not met.")
        elif result.get("reason") == "negative_validation_required_or_targets_unmet":
            print("The image passed reference proximity, but negative-image calibration is missing or its targets were not met.")
        else:
            print("The disease evidence did not meet the acceptance policy.")
        if result.get("candidate_class") is not None:
            print("Diagnostic candidate only (not an accepted diagnosis):", result["candidate_class"])
    elif status == "experimental_candidate":
        print("EXPERIMENTAL DISEASE CANDIDATE:", result["prediction"])
        print("The image passed reference similarity and the disease-score policy.")
        print("Input relevance is unverified. This result requires review and is not an accepted diagnosis.")
    elif status == "disease_detected":
        print("SUPPORTED DISEASE CANDIDATE:", result["prediction"])
    else:
        raise ValueError(f"Unrecognized prediction status: {status!r}")

    # Rejected images never receive a disease confidence or class-scores display.
    if status in {"uncertain", "experimental_candidate", "disease_detected"} and result.get("confidence_score") is not None:
        print(f"Class confidence: {result['confidence_score'] * 100:.2f}%")
        print(f"Confidence margin: {result['confidence_margin'] * 100:.2f}%")
        print("\nCLASS SCORES (comparison between supported labels only)")
        for item in result.get("top_predictions", []):
            print(f"{item['class']}: {item['confidence'] * 100:.2f}%")
    print("\nThis model cannot confirm healthy bark or verify cinnamon species.")
    if result.get("domain_distance") is not None and status != "rejected":
        print(f"Reference distance: {result['domain_distance']:.4f}; threshold: {result['domain_distance_threshold']:.4f}")
    if result.get("input_relevance_verified") is False:
        print("Input relevance is unverified; reference similarity alone cannot identify every unrelated image.")
    if result.get("domain_gate_negative_targets_met") is False:
        print("No successful negative-image calibration is recorded; unrelated-image rejection accuracy is unmeasured.")

if uploaded:
    for uploaded_filename in uploaded:
        print("\nUploaded image:", uploaded_filename)
        try:
            with Image.open(uploaded_filename) as image:
                preview = ImageOps.exif_transpose(image).convert("RGB")
                plt.figure(figsize=(7, 7))
                plt.imshow(preview)
                plt.axis("off")
                plt.show()
        except (OSError, ValueError, Image.DecompressionBombError):
            print("Preview unavailable; the prediction function will report an invalid image.")
        prediction_result = predict_bark_disease(uploaded_filename)
        display_bark_prediction(prediction_result)
else:
    print("No optional image uploaded. To test, set RUN_OPTIONAL_MANUAL_TEST=True in CELL 96 and run CELL 96 followed by CELL 97.")


## Reconstruction log

- The exact Kaggle ID and class order remain unchanged; no healthy label is introduced.
- Missing bodies and file writers are reconstructed; strings contain actual newlines.
- Stage 2 reloads an independent Stage 1 checkpoint, preventing fine-tuning from destroying the Stage 1 candidate.
- Inference has no random augmentation or external `/255`; training uses the supplied mild transformations.
- Near-duplicate auditing reports heuristic pairs with resource caps; it does not claim tree independence or silently delete data.
- Dataset caching uses disk to limit RAM pressure.
- Unmet validation policy targets disable automatic acceptance; uncertainty is preserved rather than inventing a healthy result.
- Export parity checks compare scores, labels and uncertainty decisions. Failed formats remain diagnostic in the ZIP.
- Colab versions are recorded; old backend target pins are written as documentation only.
- The optional upload is disabled by default for uninterrupted Run all.

## Per-cell record

| CELL | Supplied title | Repair |
|---:|---|---|
| 1 | Check GPU | Preserved the supplied Colab shell command. |
| 2 | Install only missing packages | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 3 | Imports | Restored imports; added importlib.metadata to record installed package versions without changing packages. |
| 4 | Print and record Colab environment | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 5 | Reproducibility settings | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 6 | Download dataset from Kaggle | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 7 | Display dataset structure | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 8 | Automatically locate the Diseases folder | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 9 | Define the exact model class order | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 10 | Build image manifest | Completed the class-count display and supplied explicit columns so an empty manifest receives a clear validation error. |
| 11 | Make sure both classes contain enough images | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 12 | Validate every image | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 13 | Remove invalid images | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 14 | Calculate SHA-256 hashes | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 15 | Find exact duplicates | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 16 | Detect duplicate images with conflicting labels | Exclude all copies of cross-label identical files from the in-memory manifest, preserve originals, and retain an audit table instead of stopping or selecting a label arbitrarily. |
| 17 | Remove duplicate copies | Completed duplicate removal and post-cleaning class checks; added a pre-deduplication guard so CELL 17 cannot silently choose a label for an unresolved cross-label file. |
| 18 | Near-duplicate audit | The source only defined a hash function. Added candidate comparison, a results table, and explicit resource-cap/completeness reporting; no automatic deletion. |
| 19 | Plot class distribution | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 20 | Display random images from each class | Completed the truncated image display loop and hid unused axes. |
| 21 | Create working directory | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 22 | Create 70/15/15 train-validation-test split | Completed test partition reporting; retained the supplied image-level split and its independence limitation. |
| 23 | Verify no exact duplicate leakage | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 24 | Save permanent split manifest | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 25 | Define exact bark preprocessing | Completed the missing tf_load_image body with static shape and label dtype; preserved EXIF/RGB/Pillow bilinear/[0,255]. |
| 26 | Define training-only augmentation | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 27 | Build TensorFlow datasets | Completed batching, training-only shuffle/augmentation and all three dataset definitions. Used disk cache and clipped augmented pixels to [0,255]. |
| 28 | Verify model input tensor | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 29 | Display augmented training samples | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 30 | Calculate class weights | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 31 | Build EfficientNetB0 model | Completed the missing pooling/dropout/two-output softmax head and model construction. No augmentation or external rescaling in the inference graph. |
| 32 | Verify model input/output contract | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 33 | Compile Stage 1 | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 34 | Stage 1 callbacks | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 35 | Train Stage 1 | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 36 | Prediction helper | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 37 | Load and evaluate best Stage 1 checkpoint | Completed the truncated validation output, retaining the best validation-loss checkpoint and recording all comparison metrics. |
| 38 | Start Stage 2 from the BEST Stage 1 model | Fixed the supplied alias model=stage1_model: reload an independent checkpoint before fine-tuning so Stage 1 remains selectable. Explicitly set both frozen and unfrozen flags. |
| 39 | Recompile Stage 2 | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 40 | Stage 2 callbacks | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 41 | Train Stage 2 | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 42 | Evaluate Stage 2 | Completed the truncated validation output using the best Stage 2 checkpoint. |
| 43 | Compare both training stages | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 44 | Automatically select the better model | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 45 | Plot Stage 1 accuracy | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 46 | Plot Stage 1 loss | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 47 | Plot Stage 2 accuracy | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 48 | Plot Stage 2 loss | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 49 | Combine and save training history | Closed the truncated training-history export call. |
| 50 | Calculate validation confidence and margin | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 51 | Search validation thresholds | Completed the truncated threshold search, accepted-accuracy/coverage calculations and missing threshold_df definition. |
| 52 | Automatically choose uncertainty threshold | Completed threshold selection and scalar definitions. Documented a conservative missing-target fallback: disable acceptance instead of claiming unmet 90%/60% goals. |
| 53 | FINAL held-out test evaluation | Completed the truncated held-out evaluation and per-class classification report; no healthy-class metrics are invented. |
| 54 | Test confusion matrix | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 55 | Apply uncertainty policy to test results | Completed truncated accepted-test metrics and handled zero accepted examples without NaN or division errors. |
| 56 | Build detailed test-prediction table | Completed the truncated result columns used by error analysis and final CSV export. |
| 57 | Show misclassifications | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 58 | Display misclassified images | Completed the truncated error-image display and empty-mistakes handling. |
| 59 | Single image preprocessing helper | Completed the truncated return tuple while retaining the same preprocessing as training. |
| 60 | Application-style prediction function | Completed margin, policy decision and application-style response. The function never returns healthy and does not claim out-of-domain rejection. |
| 61 | Test inference function | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 62 | Grad-CAM function | Completed the truncated Grad-CAM graph and gradient calculation through the nested backbone and existing classifier head. |
| 63 | Display a Grad-CAM example | Completed the missing heatmap resize/overlay; retained the model softmax output and explained the visualization limit. |
| 64 | Create final handoff directories | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 65 | Save final Keras model | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 66 | Save companion HDF5 model | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 67 | Save exact class order | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 68 | Save label definitions | Completed the truncated label-definitions JSON writer; preserved the exact two labels. |
| 69 | Save backend preprocessing.py | Restored line breaks inside the generated Python string and added the missing file write. |
| 70 | Save training environment | Completed environment data and JSON writer; recorded actual package/platform/build versions instead of installing target-backend pins. |
| 71 | Save dataset summary | Completed dataset-summary fields and writer; records excluded cross-label conflicts and exports their audit alongside near-duplicate/invalid-image audits. |
| 72 | Copy training history and split manifest | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 73 | Save classification report | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 74 | Save test predictions | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 75 | Save confusion matrix CSV and PNG | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 76 | Save threshold calibration | Completed the policy export and documented validation targets, fallback and unsupported healthy/OOD claims. |
| 77 | Save final metrics | Completed truncated metrics with actual test support/coverage; unsupported health accuracy remains null. |
| 78 | Save error analysis | Completed the truncated error-analysis writer and documented the retained split/domain limitations. |
| 79 | Create real-image parity fixture set | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 80 | Copy parity images | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 81 | Calculate original fixture predictions | Completed fixture records with image/tensor hashes, repeated-inference checks and expected uncertainty decisions. |
| 82 | Reload .keras and compare predictions | Completed the truncated difference loop and missing aggregate variables; also checks uncertainty-policy agreement. |
| 83 | Reload .h5 and compare predictions | Completed the truncated HDF5 parity loop and missing aggregate variables; also checks uncertainty-policy agreement. |
| 84 | Evaluate export parity | Completed the truncated parity status display and policy checks; reports failures honestly while allowing the diagnostic ZIP to be created. |
| 85 | Save parity information | Completed the parity JSON writer; records actual format results and explicitly leaves backend parity untested. |
| 86 | Save target CinnaAI backend requirements | Restored actual newlines between requirements. Target-backend versions are documentation only, not a Colab installation command. |
| 87 | Save actual Colab training requirements | Restored requirement-file newlines and records actual KaggleHub/Matplotlib versions used by training. |
| 88 | Calculate important SHA-256 checksums | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 89 | Create metadata.json | Completed the truncated metadata with preprocessing/output/versions/policy/hashes. No backend or deployment success is asserted before testing. |
| 90 | Create backend parity test script | Completed the truncated generated parity script and writer. Supports either export and verifies model/image/tensor hashes, scores, labels and policy decisions. |
| 91 | Create README for Codex | Completed the truncated handoff README, including results, exact input/output, failed-export handling and integration instructions. |
| 92 | Create complete file checksums | Completed the truncated checksum writer, with real newlines and portable relative paths. |
| 93 | Display final handoff package | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 94 | Create final ZIP | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 95 | Download final model package | Restored lost line breaks and Python indentation; retained the supplied operations. |
| 96 | Optional manual prediction test | Guarded the optional interactive upload so top-to-bottom Run all does not block on an unnecessary manual test. |
| 97 | Predict uploaded image | Added the empty-upload guard corresponding to the explicitly optional manual test. |

## Verification performed

- All 97 code cells retain their supplied numbering, exact titles and order, each preceded by its Markdown heading.
- Python syntax checks passed for the 96 Python cells; CELL 1 is the supplied Colab shell command. Generated preprocessing and backend parity scripts also compile.
- A local CPU smoke test executed CELLS 3–94 and the disabled optional CELLS 96–97 with 48 synthetic images, plus a corrupt image and an exact duplicate.
- The smoke environment used TensorFlow 2.16.2 and Keras 3.15.1. Only the test harness substituted offline randomly initialized EfficientNetB0 weights and one training batch/epoch per stage; the delivered notebook retains ImageNet and 25/35 epoch limits.
- Frozen-head training, selective fine-tuning, Stage 1 candidate preservation, validation selection, held-out evaluation, and Grad-CAM for frozen/fine-tuned models passed.
- Both exported formats had zero maximum score difference on the synthetic smoke fixtures. This does not predict parity for the later real ImageNet-trained model.
- The generated backend script ran successfully for both formats; preprocessing pixels, fixture/model hashes, labels and uncertainty decisions matched.
- Both threshold-target outcomes and failed-parity cases were exercised. JSON, requirement-file line breaks, checksum coverage and ZIP integrity were verified.
- Actual Kaggle access, ImageNet download, Colab GPU execution, browser download/upload and full real-dataset training were not exercised locally. They must run in Colab; no disease performance is claimed from synthetic tests.

## Cross-label duplicate correction

The reported Colab conflict for IMG_1668.JPG is handled by excluding every matching file hash from the training/evaluation manifest. Same-label deduplication in CELL 17 must not decide a conflicting label. CELL 71 exports the excluded rows and counts; original dataset files remain available for review.

Targeted checks passed for exclusion of both conflicting copies, blocking unresolved conflicts in CELL 17, same-label deduplication, the no-conflict path and CSV/summary audit export. All 96 Python cells still pass syntax checks; CELL 1 remains a Colab shell command.


## Unsupported-image rejection revision — 15 October 2026

The prior 97-cell verification record above documents the earlier notebook. The updated notebook has all 97 original sections plus 7 titled subsections (104 code cells total). It uses a training-only cosine kNN reference gate, optional labelled negative validation/test inputs and the same exported decision runtime. A high disease softmax score cannot override reference rejection. With no successful negative calibration, results inside the reference remain uncertain by default. See the overview and CELL 44A for input directories and the explicit experimental override.

Verification for this revision: 12 focused tests passed. A local synthetic CPU smoke run exercised the updated training/evaluation/export flow with 48 disease-image fixtures, 20 negative validation fixtures and 8 separate negative test fixtures. Only the smoke harness used randomly initialized EfficientNetB0 weights and one batch/epoch per stage; the notebook retains ImageNet weights and its 25/35 epoch limits. Both export probability/distance/decision parity checks and both generated backend scripts passed for that synthetic run. Checksums, reference/model binding and ZIP integrity passed. Kaggle access, a Colab GPU run, real ImageNet-trained bark rejection and browser upload/download were not tested locally.


## Domain-first/display/Drive revision

All 97 original numbered sections remain, with 8 added subsections (105 code cells total). Interactive reference rejection now precedes the disease head; rejected public results contain no disease scores. CELLS 96–97 distinguish rejected, uncertain and disease_detected. New CELL 94A optionally saves the full verified package plus a fresh ZIP to a timestamped Google Drive folder. Update the runtime definitions before testing and regenerate exports before backing up. The positive-only experiment flag remains off by default; this display correction does not validate unrelated-image rejection across real-world inputs.

### Domain-first/display/Drive verification

17 targeted tests passed, including head-skipping, all three manual-display outcomes, verified Drive-backup copying/ZIP creation on a local filesystem, tamper rejection and non-overwriting timestamped backups. The full local synthetic CPU smoke test also passed with 48 disease examples, 20 negative validation examples and 8 held-out negative examples. Both model formats, shared domain-first backend scripts, reference hashes, complete package checksums and local backup ZIP verification passed. Smoke training used offline/random EfficientNetB0 weights and one batch/epoch per stage only in the harness; the delivered notebook retains ImageNet and original training limits. Actual Colab/Drive authentication, remote Drive storage and real-world rejection accuracy were not tested locally.


## v4 verification — staged relevance and disease uncertainty

21 focused tests passed. The exact reported distance 0.33395084013745446 versus 0.27864599603105356 now yields uncertain, not rejected or an accepted disease. Two complete local synthetic CPU smoke modes passed: relevance data unavailable and relevance training enabled. The enabled mode exercised separate MobileNetV3Small head/fine-tuning training and validation selection, threshold calibration, test metrics, optional model export, both generated disease-format backend parity scripts, checksums, ZIP creation and local filesystem Drive-backup verification. Synthetic data do not establish real relevance accuracy; offline/random backbones and one batch/epoch were substitutions in the harness only. The delivered notebook retains ImageNet and its full training settings. Actual Kaggle/Colab GPU runs, remote Drive access and independent real photos are not validated locally.

The current notebook has 109 code cells: all 97 original numbered sections plus 12 labelled subsections. The older reference-only rejection and positive-only override descriptions in this historical appendix are superseded by the v4 overview and CELLS 44F–44I. With only the reported two disease folders, relevance cannot be trained; the system completes as an explicitly diagnostic/uncertain workflow rather than claiming complete input recognition.

### v5 — existing two-class workflow
Removed the mandatory missing-relevance blocker in the explicit default experimental mode. Reference-matching images reach the disease head; confident results are experimental_candidate, with review required. Metadata, metrics, fixtures, parity checks and verified Drive backups preserve this mode. No new dataset, healthy class, fabricated calibration, or threshold relaxation is introduced.
